# Kaggriculture — Training Only / Fixed Kaggle Dataset

**Purpose:** unattended BC training only. The processed dataset is already available and is **not regenerated**.

Fixed processed dataset:

`/kaggle/input/notebooks/jominurk21cs1077/preprocessing/processed`

This notebook contains all Python code required for training. It does not require a ZIP upload, external project files, or a guessed dataset location.

Execution order:

1. Materialize embedded training source.
2. Compile/import-check the complete source tree.
3. Verify the exact processed dataset path.
4. Run full dataset preflight and verify BC unit cardinality.
5. Run a real data/model forward-backward smoke test.
6. Run a 2×T4 DDP smoke test.
7. Launch unattended training with checkpoint/resume.

**No long training starts unless the gates above pass.**


In [ ]:
# ============================================================
# 0. FIXED TRAINING CONFIGURATION
# ============================================================
from pathlib import Path
import os, sys, json, shutil, subprocess, time, py_compile

PROCESSED_DIR = Path(
    "/kaggle/input/notebooks/jominurk21cs1077/preprocessing/processed"
)

WORK_ROOT = Path("/kaggle/working/kagri_training")
CKPT_DIR = WORK_ROOT / "ckpt"
MANIFEST_PATH = WORK_ROOT / "dataset_manifest.json"
CONFIG_PATH = WORK_ROOT / "train_config.json"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Effective batch = 4096 on 2×T4.
CONFIG = {
    "processed_dir": str(PROCESSED_DIR),
    "dim": 256,
    "dropout": 0.10,
    "batch_size_per_gpu": 2048,
    "workers": 4,
    "prefetch": 2,
    "cache_capacity": 2,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "warmup": 500,
    "epochs": 6,
    "patience": 3,
    "label_smoothing": 0.05,
    "return_weighting": True,
    "market_w": 0.5,
    "value_w": 0.1,
    "clip": 1.0,
    "val_max_examples": 500_000,
    "seed": 0,
    "log_every": 200,
    "save_every": 250,
}

print("Processed dataset:", PROCESSED_DIR)
print("Work directory:", WORK_ROOT)
print("Checkpoint directory:", CKPT_DIR)
print("Effective batch:", CONFIG["batch_size_per_gpu"] * 2)


In [ ]:
# ============================================================
# 1. MATERIALIZE ALL TRAINING SOURCE CODE
# ============================================================
SOURCES = {'bootstrap_kaggle.py': '"""Kaggle bootstrap for the unattended V3 run.\n\nIt finds the uploaded V3 archive under /kaggle/input, extracts it to a stable\nworking directory, validates that the Python package is present, and resolves\nthe processed dataset only when it is uniquely identifiable.\n"""\nfrom __future__ import annotations\nimport glob, os, shutil, zipfile\n\nROOT = "/kaggle/working/kagri_v3"\nINPUT = "/kaggle/input"\n\ndef find_code_zip():\n    candidates = []\n    for p in glob.glob(INPUT + "/**/*.zip", recursive=True):\n        try:\n            with zipfile.ZipFile(p) as z:\n                names = set(z.namelist())\n                if "run_bc.py" in names and "verify_codebase.py" in names and "kagri/" in names:\n                    candidates.append(p)\n        except Exception:\n            pass\n    return candidates\n\ndef find_code_root():\n    candidates = []\n    for p in glob.glob(INPUT + "/**/run_bc.py", recursive=True):\n        root = os.path.dirname(p)\n        if os.path.exists(os.path.join(root, "verify_codebase.py")) and os.path.isdir(os.path.join(root, "kagri")):\n            candidates.append(root)\n    return candidates\n\ndef find_processed():\n    preferred = os.environ.get(\n        "KAGRI_PROCESSED_DIR",\n        "/kaggle/input/notebooks/jominurk21cs1077/preprocessing/processed",\n    )\n    if os.path.isfile(os.path.join(preferred, "index.npz")):\n        return preferred\n\n    candidates = sorted(set(\n        os.path.dirname(p)\n        for p in glob.glob(INPUT + "/**/index.npz", recursive=True)\n        if os.path.isfile(p)\n    ))\n    if len(candidates) == 1:\n        return candidates[0]\n    if not candidates:\n        raise RuntimeError(\n            "No processed dataset containing index.npz was found under /kaggle/input. "\n            "Mount the processed Kaggriculture dataset first."\n        )\n    raise RuntimeError(\n        "Multiple processed datasets were found. No guess was made. "\n        "Set KAGRI_PROCESSED_DIR to the exact directory. Candidates:\\n" +\n        "\\n".join(candidates)\n    )\n\ndef install_code():\n    shutil.rmtree(ROOT, ignore_errors=True)\n    os.makedirs(ROOT, exist_ok=True)\n\n    roots = find_code_root()\n    zips = find_code_zip()\n\n    if len(roots) == 1:\n        shutil.copytree(roots[0], ROOT, dirs_exist_ok=True)\n        source = roots[0]\n    elif len(roots) > 1:\n        raise RuntimeError("Multiple V3 code roots found; refusing to guess:\\n" + "\\n".join(roots))\n    elif len(zips) == 1:\n        with zipfile.ZipFile(zips[0]) as z:\n            z.extractall(ROOT)\n        source = zips[0]\n    elif len(zips) > 1:\n        raise RuntimeError("Multiple V3 ZIPs found; refusing to guess:\\n" + "\\n".join(zips))\n    else:\n        raise RuntimeError(\n            "Could not find V3 code. Upload the kaggriculture_v3_evidence_backed.zip "\n            "as a Kaggle dataset/input."\n        )\n\n    if not os.path.exists(os.path.join(ROOT, "run_bc.py")):\n        raise RuntimeError("Bootstrap extraction succeeded but run_bc.py is missing.")\n    if not os.path.isdir(os.path.join(ROOT, "kagri")):\n        raise RuntimeError("Bootstrap extraction succeeded but kagri/ is missing.")\n\n    processed = find_processed()\n    os.environ["KAGRI_PROCESSED_DIR"] = processed\n\n    print("=== KAGRI V3 BOOTSTRAP ===")\n    print("Code source:", source)\n    print("Code root:", ROOT)\n    print("Processed dataset:", processed)\n    print("==========================")\n    return ROOT, processed\n\nif __name__ == "__main__":\n    install_code()\n', 'run_bc.py': '"""Entry point for the long unattended BC run.\n\nKaggle notebook command:\n    !torchrun --standalone --nproc_per_node=2 run_bc.py\n\nFor a single GPU:\n    !python run_bc.py\n"""\nfrom kagri.configs import BC\nfrom kagri.train import train_bc\n\nif __name__ == "__main__":\n    train_bc(BC)\n', 'verify_codebase.py': '"""Fast offline verification of the codebase without the Kaggriculture dataset."""\nimport os, tempfile\nimport numpy as np\nimport torch\nfrom kagri.encoding import expand_grid_numpy\nfrom kagri.datasets import expand_grid_torch, UnitTransitionDataset, ShardBatchSampler\n\n\ndef test_grid_equivalence():\n    rng=np.random.default_rng(0)\n    x=np.zeros((2,2,10,10,6),dtype=np.uint8)\n    x[...,0]=rng.integers(0,6,size=x[...,0].shape)\n    x[...,1]=rng.integers(0,6,size=x[...,1].shape)\n    x[...,2]=rng.integers(0,4,size=x[...,2].shape)\n    x[...,3]=rng.integers(0,16,size=x[...,3].shape)\n    x[...,4]=rng.integers(0,128,size=x[...,4].shape)\n    x[...,5]=rng.integers(0,256,size=x[...,5].shape)\n    # The reference encoder expects one observation grid at a time.\n    ref=np.stack([expand_grid_numpy(x[i]) for i in range(x.shape[0])])\n    got=expand_grid_torch(torch.from_numpy(x)).cpu().float().numpy()\n    err=np.max(np.abs(ref-got))\n    assert err==0.0, f"grid mismatch: max error {err}"\n    print("PASS grid expansion: exact")\n\n\ndef test_dataset_count_semantics():\n    with tempfile.TemporaryDirectory() as d:\n        n_turns=4\n        counts=np.array([1,3,2,4],dtype=np.int16)\n        total=int(counts.sum())\n        data={\n            "grids":np.zeros((n_turns,2,10,10,6),np.uint8),\n            "scalars":np.zeros((n_turns,59),np.float16),\n            "unit_ops":np.arange(total,dtype=np.int16),\n            "unit_counts":counts,\n            "market_qty":np.zeros((n_turns,19),np.int8),\n            "market_bin":np.zeros((n_turns,2),np.int8),\n            "rewards":np.zeros(n_turns,np.float32),\n            "episode_id":np.arange(n_turns,dtype=np.int64),\n            "player":np.zeros(n_turns,np.int8),\n            "step":np.arange(n_turns,dtype=np.int16),\n            "final_reward":np.zeros(n_turns,np.float32),\n            "win":np.zeros(n_turns,np.float16),\n        }\n        np.savez_compressed(os.path.join(d,"shard_1.npz"),**data)\n        np.savez(os.path.join(d,"index.npz"),shards=np.array(["shard_1.npz"]),counts=np.array([n_turns]))\n        ds=UnitTransitionDataset(d,"train",val_frac=0.0)\n        assert len(ds)==total, (len(ds),total)\n        b=ds.__getitems__(list(range(total)))\n        assert b["unit_op"].shape[0]==total\n        print("PASS dataset semantics: unit count is authoritative")\n\n\ndef test_sampler_partition():\n    shards=[("a",10,10),("b",10,14),("c",10,8)]\n    a=ShardBatchSampler(shards,4,True,0,0,2,True); b=ShardBatchSampler(shards,4,True,0,1,2,True)\n    aa=[x for batch in a for x in batch]; bb=[x for batch in b for x in batch]\n    assert len(aa)==len(bb)==a.__len__()*4\n    assert set(aa).isdisjoint(bb)\n    assert len(set(aa+bb))==len(aa+bb)\n    print("PASS DDP batch partition: disjoint and balanced")\n\n\nif __name__=="__main__":\n    test_grid_equivalence()\n    test_dataset_count_semantics()\n    test_sampler_partition()\n    print("ALL OFFLINE TESTS PASSED")\n', 'main.py': '"""Kaggriculture competition agent.\n\nLoads bc_best.pt/bc_last.pt and uses the exact shared encoder used during\npreprocessing.  The legality mask remains the final safety layer.\n"""\nfrom __future__ import annotations\nimport os\nimport numpy as np\n\n_HERE=os.path.dirname(os.path.abspath(__file__))\n_MODEL=None; _DEV=None\ntry:\n    import torch\n    from kagri.models import BCPolicy\n    from kagri.datasets import expand_grid_torch\n    _TORCH_OK=True\nexcept Exception as e:\n    _TORCH_OK=False\n    print("Neural stack unavailable; heuristic fallback:",e)\n\nfrom kagri.encoding import encode_observation, decode_unit_op, assemble_market, legal_unit_op_mask\n\n\ndef _find_ckpt():\n    for name in ("bc_best.pt","bc_last.pt","model.pt"):\n        p=os.path.join(_HERE,name)\n        if os.path.exists(p): return p\n    return None\n\n\ndef _load_model():\n    global _DEV\n    if not _TORCH_OK: return None\n    p=_find_ckpt()\n    if p is None: return None\n    _DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    ckpt=torch.load(p,map_location=_DEV,weights_only=False)\n    cfg=ckpt.get("cfg",{})\n    model=BCPolicy(cfg.get("dim",256),0.0,cfg.get("spatial_pool",2)).to(_DEV)\n    model.load_state_dict(ckpt["model"]); model.eval()\n    print(f"Loaded {os.path.basename(p)} on {_DEV}")\n    return model\n\n\ndef _prep(obs,player):\n    grid,scal=encode_observation(obs,player)\n    g=torch.from_numpy(grid[None]).to(_DEV)\n    g=expand_grid_torch(g).float()\n    s=torch.from_numpy(scal[None].astype(np.float32)).to(_DEV)\n    return g,s\n\n\ndef _unit(model,g,s,feat,obs,player,xy):\n    uf=torch.from_numpy(np.asarray(feat,dtype=np.float32)[None]).to(_DEV)\n    logits=model(g,s,uf)[0][0].float().cpu().numpy()\n    mask=legal_unit_op_mask(obs,player,xy)\n    logits=np.where(mask>0,logits,-1e9)\n    return int(np.argmax(logits))\n\n\ndef _market(model,g,s):\n    dummy=torch.zeros((1,2),dtype=torch.float16,device=_DEV)\n    _,qty,binl,_=model(g,s,dummy)\n    return assemble_market(qty[0].argmax(-1).cpu().numpy(),binl[0].argmax(-1).cpu().numpy(),max_orders=10)\n\n\ndef _heuristic(obs):\n    player=int(obs.get("player",0)); me=obs["farms"][player]; priv=obs.get("private",{})\n    fx,fy=me["farmer"]; tile=me.get("tiles",[[None]])[fy][fx]\n    market=[]\n    if priv.get("seeds",{}).get("WHEAT",0)==0 and me.get("money",0)>=10: market.append(["BUY_SEED","WHEAT",1])\n    wheat=priv.get("shed",{}).get("WHEAT",0)\n    if wheat>0: market.append(["SELL","WHEAT",wheat])\n    if tile is None and priv.get("seeds",{}).get("WHEAT",0)>0: return {"farmer":["PLANT","WHEAT"],"hands":[],"market":market}\n    if isinstance(tile,dict) and tile.get("kind")=="PLANT":\n        if obs.get("day",0)-tile.get("planted_day",0)>=2: return {"farmer":["HARVEST"],"hands":[],"market":market}\n        if not tile.get("watered_today"): return {"farmer":["WATER"],"hands":[],"market":market}\n    return {"farmer":["PASS"],"hands":[],"market":market}\n\n\ndef agent(obs):\n    global _MODEL\n    if _TORCH_OK and _MODEL is None: _MODEL=_load_model()\n    if _MODEL is None: return _heuristic(obs)\n    player=int(obs.get("player",0)); me=obs["farms"][player]; g,s=_prep(obs,player)\n    fx,fy=me["farmer"]\n    farmer=decode_unit_op(_unit(_MODEL,g,s,[1.0,0.0],obs,player,(fx,fy)))\n    hands=[]\n    for i,h in enumerate(me.get("hands",[]) or []):\n        hx,hy=h[:2]; hands.append(decode_unit_op(_unit(_MODEL,g,s,[0.0,(i+1)/12.0],obs,player,(hx,hy))))\n    return {"farmer":farmer,"hands":hands,"market":_market(_MODEL,g,s)}\n', 'kagri/datasets.py': '"""Fast, correct dataset layer for the Kaggriculture processed shards.\n\nCritical correctness rule:\n    index.npz ``counts`` is a count of TURN ROWS, while a BC example is one\n    UNIT within a turn.  Therefore the BC dataset length is\n    sum(unit_counts), not sum(index.counts).\n\nThe original V2 code treated turn-row counts as unit-example counts.  That is\nwhy the observed run reported 6,619,680 examples instead of the ~59.7M unit\nexamples from the earlier run.  This version detects and fixes that mismatch.\n\nFor throughput, batches are generated shard-locally.  A batch therefore reads\none shard, keeping the per-worker LRU cache hot and avoiding a global random\nindex materialization.  Within each shard the unit order is shuffled each epoch.\n"""\nfrom __future__ import annotations\nimport os\nfrom collections import OrderedDict\nfrom dataclasses import dataclass\nfrom typing import Iterator\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset, Sampler\n\nfrom .encoding import CROPS, ANIMALS, GRID_CHANNELS\n\nTOTAL_GRID_CHANNELS = 2 * GRID_CHANNELS\n\n# --------------------------------------------------------------------------- #\n# Packed-grid expansion\n# --------------------------------------------------------------------------- #\n_EXPAND_TABLES = {}\n\n\ndef _build_expand_tables(device: torch.device, dtype: torch.dtype):\n    key = (str(device), dtype)\n    if key in _EXPAND_TABLES:\n        return _EXPAND_TABLES[key]\n\n    C = GRID_CHANNELS\n    tables = []\n    for field in range(6):\n        t = torch.zeros(256, C, device=device, dtype=dtype)\n        if field == 0:\n            t[:6, :6] = torch.eye(6, device=device, dtype=dtype)\n        elif field == 1:\n            for k in range(1, len(CROPS) + 1):\n                t[k, 6 + k - 1] = 1\n        elif field == 2:\n            base = 6 + len(CROPS)\n            for k in range(1, len(ANIMALS) + 1):\n                t[k, base + k - 1] = 1\n        elif field == 3:\n            base = 6 + len(CROPS) + len(ANIMALS)\n            t[:, base] = torch.arange(256, device=device, dtype=dtype).clamp_max(15) / 15\n        elif field == 4:\n            base = 6 + len(CROPS) + len(ANIMALS) + 1\n            vals = torch.arange(256, device=device, dtype=torch.int64).unsqueeze(1)\n            bits = ((vals >> torch.arange(7, device=device)) & 1).to(dtype)\n            t[:, base:base + 7] = bits\n        else:\n            base = 6 + len(CROPS) + len(ANIMALS) + 1 + 7\n            vals = torch.arange(256, device=device, dtype=dtype)\n            t[:, base] = vals.remainder(16) / 15\n            t[:, base + 1] = torch.floor(vals / 16) / 15\n        tables.append(t)\n    _EXPAND_TABLES[key] = tuple(tables)\n    return _EXPAND_TABLES[key]\n\n\ndef expand_grid_torch(codes: torch.Tensor) -> torch.Tensor:\n    """Expand uint8 [B,2,10,10,6] to [B,48,10,10]."""\n    dtype = torch.float16 if codes.is_cuda else torch.float32\n    tables = _build_expand_tables(codes.device, dtype)\n    x = codes.long()\n    farms = []\n    for f in range(2):\n        g = x[:, f]\n        z = (tables[0][g[..., 0]] + tables[1][g[..., 1]] +\n             tables[2][g[..., 2]] + tables[3][g[..., 3]] +\n             tables[4][g[..., 4]] + tables[5][g[..., 5]])\n        farms.append(z.permute(0, 3, 1, 2).contiguous())\n    return torch.cat(farms, dim=1)\n\n\n# --------------------------------------------------------------------------- #\n# Shard metadata / cache\n# --------------------------------------------------------------------------- #\nclass _ShardCache:\n    def __init__(self, out_dir: str, capacity: int = 8):\n        self.out_dir = out_dir\n        self.capacity = capacity\n        self.cache = OrderedDict()\n\n    def get(self, shard_name: str):\n        if shard_name in self.cache:\n            self.cache.move_to_end(shard_name)\n            return self.cache[shard_name]\n        path = os.path.join(self.out_dir, shard_name)\n        with np.load(path, allow_pickle=False) as z:\n            data = {k: z[k] for k in z.files}\n        counts = data["unit_counts"].astype(np.int64, copy=False)\n        data["_unit_offsets"] = np.concatenate(\n            (np.array([0], dtype=np.int64), np.cumsum(counts, dtype=np.int64))\n        )\n        data["_turn_cum"] = data["_unit_offsets"][1:]\n        data["_unit_total"] = int(counts.sum())\n        self.cache[shard_name] = data\n        self.cache.move_to_end(shard_name)\n        while len(self.cache) > self.capacity:\n            self.cache.popitem(last=False)\n        return data\n\n\ndef _read_index(out_dir: str):\n    path = os.path.join(out_dir, "index.npz")\n    if not os.path.exists(path):\n        raise FileNotFoundError(f"Missing processed index: {path}")\n    with np.load(path, allow_pickle=False) as idx:\n        shards = list(idx["shards"].astype(str))\n        turn_counts = idx["counts"].astype(np.int64)\n        unit_counts = idx["unit_counts"].astype(np.int64) if "unit_counts" in idx.files else None\n    if len(shards) != len(turn_counts):\n        raise ValueError("index.npz shards/counts length mismatch")\n    return shards, turn_counts, unit_counts\n\n\ndef _split_shards(out_dir: str, val_frac=0.05, seed=0):\n    shards, turn_counts, indexed_unit_counts = _read_index(out_dir)\n    # Derive authoritative unit totals from each shard.  This is intentionally\n    # done once at dataset construction and is tiny compared with the examples.\n    cache = _ShardCache(out_dir, capacity=max(8, len(shards)))\n    unit_counts = []\n    for i, name in enumerate(shards):\n        d = cache.get(name)\n        actual_turns = int(d["unit_counts"].shape[0])\n        if actual_turns != int(turn_counts[i]):\n            raise ValueError(\n                f"Index mismatch for {name}: index says {turn_counts[i]} turn rows, "\n                f"shard contains {actual_turns}. Regenerate index.npz."\n            )\n        unit_counts.append(int(d["_unit_total"]))\n    unit_counts = np.asarray(unit_counts, dtype=np.int64)\n\n    rng = np.random.default_rng(seed)\n    order = rng.permutation(len(shards))\n    n_val = max(1, int(len(shards) * val_frac)) if len(shards) > 1 else 0\n    val_ids = set(order[:n_val].tolist())\n    train = [(shards[i], int(turn_counts[i]), int(unit_counts[i]))\n             for i in range(len(shards)) if i not in val_ids]\n    val = [(shards[i], int(turn_counts[i]), int(unit_counts[i]))\n           for i in range(len(shards)) if i in val_ids]\n    return train, val\n\n\n# --------------------------------------------------------------------------- #\n# Shard-local batch sampler\n# --------------------------------------------------------------------------- #\n@dataclass\nclass ShardBatchSampler(Sampler[list[int]]):\n    shards: list[tuple[str, int, int]]\n    batch_size: int\n    shuffle: bool = True\n    seed: int = 0\n    rank: int = 0\n    world_size: int = 1\n    drop_last: bool = True\n\n    def __post_init__(self):\n        self.epoch = 0\n        self.starts = []\n        s = 0\n        for _, _, units in self.shards:\n            self.starts.append(s)\n            s += int(units)\n        self.total_units = s\n        total_batches = sum(int(u) // self.batch_size for _, _, u in self.shards)\n        self._usable_batches = total_batches - (total_batches % self.world_size)\n\n    def set_epoch(self, epoch: int):\n        self.epoch = int(epoch)\n\n    def __len__(self):\n        return self._usable_batches // self.world_size\n\n    def __iter__(self) -> Iterator[list[int]]:\n        rng = np.random.default_rng(self.seed + 1009 * self.epoch)\n        shard_order = np.arange(len(self.shards))\n        if self.shuffle:\n            rng.shuffle(shard_order)\n\n        global_batch_id = 0\n        emitted = 0\n        for si in shard_order:\n            _, _, n_units = self.shards[int(si)]\n            n_units = int(n_units)\n            n_full = n_units // self.batch_size\n            if n_full <= 0:\n                continue\n            local = np.arange(n_units, dtype=np.int64)\n            if self.shuffle:\n                rng.shuffle(local)\n            start_global = self.starts[int(si)]\n            for b in range(n_full):\n                if global_batch_id >= self._usable_batches:\n                    return\n                if global_batch_id % self.world_size == self.rank:\n                    a = b * self.batch_size\n                    batch = (local[a:a + self.batch_size] + start_global).tolist()\n                    yield batch\n                    emitted += 1\n                global_batch_id += 1\n        assert emitted == len(self), (emitted, len(self))\n\n\n# --------------------------------------------------------------------------- #\n# Unit transition dataset\n# --------------------------------------------------------------------------- #\nclass UnitTransitionDataset(Dataset):\n    """One example per unit action, with vectorized batched fetching."""\n    def __init__(self, out_dir, split="train", val_frac=0.05, seed=0,\n                 cache_capacity=8):\n        self.out_dir = out_dir\n        tr, va = _split_shards(out_dir, val_frac, seed)\n        self.shards = tr if split == "train" else va\n        self.cache = _ShardCache(out_dir, cache_capacity)\n        self._shard_lengths = np.asarray([x[2] for x in self.shards], dtype=np.int64)\n        self._shard_cum = np.cumsum(self._shard_lengths, dtype=np.int64)\n        self._total = int(self._shard_cum[-1]) if len(self._shard_cum) else 0\n        self._turn_cums = [self.cache.get(name)["_turn_cum"] for name, _, _ in self.shards]\n\n    def __len__(self):\n        return self._total\n\n    def _resolve_indices(self, indices):\n        idx = np.asarray(indices, dtype=np.int64)\n        shard_ids = np.searchsorted(self._shard_cum, idx, side="right")\n        prev_shard = np.zeros_like(idx)\n        m = shard_ids > 0\n        prev_shard[m] = self._shard_cum[shard_ids[m] - 1]\n        local = idx - prev_shard\n        turns = np.empty_like(idx)\n        units = np.empty_like(idx)\n        for si in np.unique(shard_ids):\n            mask = shard_ids == si\n            tc = self._turn_cums[int(si)]\n            t = np.searchsorted(tc, local[mask], side="right")\n            prev = np.zeros_like(t)\n            mm = t > 0\n            prev[mm] = tc[t[mm] - 1]\n            turns[mask] = t\n            units[mask] = local[mask] - prev\n        return shard_ids, turns, units\n\n    def __getitem__(self, i):\n        b = self.__getitems__([int(i)])\n        return {k: (v[0] if torch.is_tensor(v) else v) for k, v in b.items()}\n\n    def __getitems__(self, indices):\n        if not indices:\n            raise ValueError("Empty batch requested")\n        shard_ids, turns, units = self._resolve_indices(indices)\n        n = len(indices)\n        first = self.cache.get(self.shards[int(shard_ids[0])][0])\n        grid = np.empty((n,) + first["grids"].shape[1:], dtype=np.uint8)\n        scal = np.empty((n,) + first["scalars"].shape[1:], dtype=np.float16)\n        mqty = np.empty((n,) + first["market_qty"].shape[1:], dtype=np.int8)\n        mbin = np.empty((n,) + first["market_bin"].shape[1:], dtype=np.int8)\n        unit_op = np.empty(n, dtype=np.int16)\n        reward = np.empty(n, dtype=np.float32)\n        final_r = np.empty(n, dtype=np.float32)\n        win = np.empty(n, dtype=np.float16)\n        turn_units = np.empty(n, dtype=np.float16)\n\n        for si in np.unique(shard_ids):\n            pos = np.flatnonzero(shard_ids == si)\n            d = self.cache.get(self.shards[int(si)][0])\n            tt = turns[pos]\n            uu = units[pos]\n            grid[pos] = d["grids"][tt]\n            scal[pos] = d["scalars"][tt]\n            mqty[pos] = d["market_qty"][tt]\n            mbin[pos] = d["market_bin"][tt]\n            offs = d["_unit_offsets"][tt]\n            unit_op[pos] = d["unit_ops"][offs + uu]\n            reward[pos] = d["rewards"][tt]\n            final_r[pos] = d["final_reward"][tt]\n            win[pos] = d["win"][tt]\n            turn_units[pos] = d["unit_counts"][tt]\n\n        return {\n            "grid_codes": torch.from_numpy(grid),\n            "scalar": torch.from_numpy(scal),\n            "unit_feat": torch.from_numpy(\n                np.stack(((units == 0).astype(np.float16),\n                          (units.astype(np.float32) / 12.0).astype(np.float16)), axis=1)\n            ),\n            "unit_op": torch.from_numpy(unit_op),\n            "market_qty": torch.from_numpy(mqty),\n            "market_bin": torch.from_numpy(mbin),\n            "reward": torch.from_numpy(reward),\n            "final_reward": torch.from_numpy(final_r),\n            "win": torch.from_numpy(win),\n            "turn_weight": torch.from_numpy(1.0 / np.maximum(turn_units, 1.0)),\n        }\n\n\ndef collate_unit(batch):\n    if isinstance(batch, dict):\n        return batch\n    raise TypeError("UnitTransitionDataset should be fetched in batched mode")\n\n\n# --------------------------------------------------------------------------- #\n# Decision Transformer dataset (kept compatible, but correctness-first)\n# --------------------------------------------------------------------------- #\nclass TrajectoryDataset(Dataset):\n    def __init__(self, out_dir, context=40, gamma=1.0, split="train",\n                 val_frac=0.05, seed=0, cache_capacity=6):\n        self.out_dir = out_dir\n        self.context = context\n        self.gamma = gamma\n        tr, va = _split_shards(out_dir, val_frac, seed)\n        self.shards = tr if split == "train" else va\n        self.cache = _ShardCache(out_dir, cache_capacity)\n        self.trajs = []\n        for si, (name, _, _) in enumerate(self.shards):\n            d = self.cache.get(name)\n            groups = {}\n            for row, (ep, pl, st) in enumerate(zip(d["episode_id"], d["player"], d["step"])):\n                groups.setdefault((int(ep), int(pl)), []).append(row)\n            for rows in groups.values():\n                rows.sort(key=lambda r: int(d["step"][r]))\n                if len(rows) >= 2:\n                    self.trajs.append((si, rows))\n\n    def __len__(self):\n        return len(self.trajs)\n\n    def _rtg(self, rewards):\n        rtg = np.zeros_like(rewards, dtype=np.float32)\n        acc = 0.0\n        for i in range(len(rewards) - 1, -1, -1):\n            acc = rewards[i] + self.gamma * acc\n            rtg[i] = acc\n        return rtg\n\n    def __getitem__(self, i):\n        si, rows = self.trajs[i]\n        d = self.cache.get(self.shards[si][0])\n        rows = np.asarray(rows)\n        L = len(rows)\n        start = 0 if L <= self.context else np.random.randint(0, L - self.context + 1)\n        sel = rows[start:start + self.context]\n        grids = d["grids"][sel]\n        scal = d["scalars"][sel].astype(np.float32)\n        rtg_full = self._rtg(d["rewards"][rows].astype(np.float32))\n        rtg = rtg_full[start:start + len(sel)]\n        offs = d["_unit_offsets"]\n        farmer_ops = d["unit_ops"][offs[sel]].astype(np.int64)\n        w = len(sel); pad = self.context - w\n        mask = np.concatenate([np.ones(w), np.zeros(pad)]).astype(np.float32)\n        if pad > 0:\n            grids = np.concatenate([grids, np.zeros((pad,) + grids.shape[1:], grids.dtype)])\n            scal = np.concatenate([scal, np.zeros((pad, scal.shape[1]), scal.dtype)])\n            rtg = np.concatenate([rtg, np.zeros(pad, np.float32)])\n            farmer_ops = np.concatenate([farmer_ops, np.zeros(pad, np.int64)])\n        return {\n            "grid": torch.from_numpy(grids),\n            "scalar": torch.from_numpy(scal),\n            "rtg": torch.from_numpy(rtg),\n            "action": torch.from_numpy(farmer_ops),\n            "timestep": torch.arange(self.context, dtype=torch.long),\n            "mask": torch.from_numpy(mask),\n        }\n\n\ndef collate_traj(batch):\n    return {\n        "grid_codes": torch.stack([b["grid"] for b in batch]).to(torch.uint8),\n        "scalar": torch.stack([b["scalar"] for b in batch]),\n        "rtg": torch.stack([b["rtg"] for b in batch]),\n        "action": torch.stack([b["action"] for b in batch]),\n        "timestep": torch.stack([b["timestep"] for b in batch]),\n        "mask": torch.stack([b["mask"] for b in batch]),\n    }\n', 'kagri/__init__.py': '"""Kaggriculture evidence-backed training package."""\n__version__ = "3.0.0"\n', 'kagri/encoding.py': '"""\nkagri.encoding\n==============\nSINGLE SOURCE OF TRUTH for turning a Kaggriculture observation/action into\nfixed-size numeric tensors, and back again at inference time.\n\nWhy one file?  The preprocessing pass (offline, on the 140 GiB replay dump) and\nthe live agent (main.py, inside kaggle-environments) MUST encode observations in\n*exactly* the same way, or the model sees a different input distribution at test\ntime than it trained on.  Both import from here, so they can never drift.\n\nEverything in this module is pure-numpy / pure-python (no torch), so it runs in\nthe preprocessing worker processes and inside the competition sandbox.\n\nThe schema this file targets is the one documented in README.md / AGENTS.md:\n  observation keys : day, hour, step, player, farms, market, town, private,\n                     remainingOverageTime\n  action keys      : farmer, hands, market\n\nDesign (kept deliberately compact so 4,824 episodes fit under Kaggle\'s 20 GB\nworking-dir limit):\n\n  * The 10x10 board of BOTH farms is packed into a uint8 tensor of shape\n    [2, 10, 10, TILE_BYTES].  Categorical fields (tile type / crop / animal)\n    are stored as small integer codes and expanded to one-hot channels *on the\n    GPU* at train time (see kagri.datasets), which keeps disk + RAM small.\n  * The scalar/global part of the observation is a float16 vector.\n\nWe deliberately split the policy into two heads that mirror how the engine\nactually consumes actions:\n  * a PER-UNIT policy  : farmer + each hired hand each emit one op  -> UNIT_OPS\n  * a MARKET  policy   : the ordered market list -> structured "intent" vector\nThis is faithful to the action schema and, crucially, deployable: at inference\nwe call the unit policy once per controlled unit and assemble the market list\nfrom the market head.\n"""\nfrom __future__ import annotations\nimport numpy as np\n\n# --------------------------------------------------------------------------- #\n#  Vocabularies (order is FROZEN — changing it invalidates preprocessed data). #\n# --------------------------------------------------------------------------- #\nCROPS   = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]\nANIMALS = ["GOOSE", "COW", "SHEEP"]\n# Products that trade on the market (sell prices/inventory are quoted for these).\nPRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON",\n            "EGG", "MILK", "WOOL", "FERTILIZER"]\nSHOPS = ["BAKERY", "PIZZA_SHOP", "BRUNCH_SPOT", "YARN_STORE",\n         "ICE_CREAM_SHOP", "PET_CAFE", "SMOOTHIE_SHOP", "FARMERS_MARKET"]\n# Non-seed shed items we track a count for.\nSHED_ITEMS = PRODUCTS + ANIMALS  # produce + fertilizer + live animals in storage\n\nQUADS = ["NW", "NE", "SW", "SE"]\n\n# Base prices used to normalise the market price vector (from README price table).\nBASE_PRICE = {"WHEAT": 25, "CARROT": 35, "TOMATO": 60, "STRAWBERRY": 120,\n              "MELON": 250, "EGG": 50, "MILK": 160, "WOOL": 200,\n              "FERTILIZER": 100}\n\n# --------------------------------------------------------------------------- #\n#  Per-unit op vocabulary.                                                     #\n#  We flatten op + its primary categorical arg into one label space so the    #\n#  per-unit policy is a single softmax (simpler + trains better than nested).  #\n# --------------------------------------------------------------------------- #\n_SIMPLE_OPS = [\n    "PASS", "NORTH", "SOUTH", "EAST", "WEST",\n    "WATER", "HARVEST", "FERTILIZE",\n    "BUILD_COOP", "BUILD_PASTURE",\n    "FEED", "COLLECT_FERTILIZER", "CARE",\n    "DIG", "DROP",\n]\n# Parameterised ops get one label per (op, arg).\n_PLANT_OPS   = [f"PLANT_{c}"   for c in CROPS]     # PLANT <crop>\n_PICKUP_OPS  = [f"PICKUP_{i}"  for i in SHED_ITEMS]  # PICKUP <item>  (n=1)\n_PLACE_OPS   = [f"PLACE_{a}"   for a in ANIMALS]   # PLACE <animal> onto structure\n\nUNIT_OPS = _SIMPLE_OPS + _PLANT_OPS + _PICKUP_OPS + _PLACE_OPS\nUNIT_OP_INDEX = {name: i for i, name in enumerate(UNIT_OPS)}\nN_UNIT_OPS = len(UNIT_OPS)\n\n# --------------------------------------------------------------------------- #\n#  Market "intent" head layout.                                               #\n#  Instead of predicting the raw ordered list (hard, variable length) we      #\n#  predict a structured intent that is trivially assembled into a legal        #\n#  ordered market list at inference (respecting maxMarketOrdersPerTurn).       #\n#                                                                             #\n#  Quantities are bucketised so the head is classification, not regression.   #\n# --------------------------------------------------------------------------- #\nQTY_BUCKETS = [0, 1, 2, 5, 10, 25, 100]      # index -> representative quantity\nN_QTY = len(QTY_BUCKETS)\n\ndef qty_to_bucket(n: int) -> int:\n    """Map an integer quantity to the nearest-but-not-greater bucket index."""\n    b = 0\n    for i, v in enumerate(QTY_BUCKETS):\n        if n >= v:\n            b = i\n    return b\n\n# Market head sub-fields (each is a categorical over N_QTY, unless noted):\n#   sell_<product>        : how much of each product to SELL          (9 heads)\n#   buy_wheat / buy_fert  : BUY_PRODUCT quantity                      (2 heads)\n#   buy_seed_<crop>       : BUY_SEED quantity                         (5 heads)\n#   buy_animal_<animal>   : BUY_ANIMAL quantity                       (3 heads)\n#   hire                  : binary (0/1)                              (1 head, 2-way)\n#   buy_land              : binary (0/1)                              (1 head, 2-way)\nMARKET_QTY_FIELDS = (\n    [f"SELL_{p}"       for p in PRODUCTS] +\n    ["BUYP_WHEAT", "BUYP_FERTILIZER"] +\n    [f"BUYSEED_{c}"    for c in CROPS] +\n    [f"BUYANIMAL_{a}"  for a in ANIMALS]\n)\nMARKET_BIN_FIELDS = ["HIRE", "BUY_LAND"]\nMARKET_FIELD_INDEX = {name: i for i, name in enumerate(MARKET_QTY_FIELDS)}\n\n# --------------------------------------------------------------------------- #\n#  Grid packing.                                                              #\n# --------------------------------------------------------------------------- #\nBOARD = 10\nTILE_BYTES = 6\n# byte0 tile_type : 0 empty,1 locked,2 weed,3 plant,4 coop,5 pasture\n# byte1 crop      : 0 none, 1..5 -> CROPS\n# byte2 animal    : 0 none, 1 goose, 2 cow, 3 sheep\n# byte3 yield_units clamped 0..15\n# byte4 flags bitfield:\n#        bit0 watered_today, bit1 fertilized(active), bit2 fed_today,\n#        bit3 cared_today,    bit4 fertilizer_available,\n#        bit5 unit_farmer_here, bit6 unit_hand_here\n# byte5 misc : low nibble consecutive_unwatered/unfed (0..15),\n#              high nibble pending_care_bonus (0..15)\nTILE_EMPTY, TILE_LOCKED, TILE_WEED, TILE_PLANT, TILE_COOP, TILE_PASTURE = range(6)\n\n_CROP_CODE   = {c: i + 1 for i, c in enumerate(CROPS)}\n_ANIMAL_CODE = {a: i + 1 for i, a in enumerate(ANIMALS)}\n\n\ndef _clamp_u8(x: int, hi: int = 255) -> int:\n    if x < 0:\n        return 0\n    return hi if x > hi else int(x)\n\n\ndef _pack_farm(farm: dict) -> np.ndarray:\n    """Pack one farm dict into a [10,10,TILE_BYTES] uint8 array."""\n    out = np.zeros((BOARD, BOARD, TILE_BYTES), dtype=np.uint8)\n    tiles = farm.get("tiles") or []\n    for y in range(min(BOARD, len(tiles))):\n        row = tiles[y]\n        for x in range(min(BOARD, len(row))):\n            t = row[x]\n            b = out[y, x]\n            if t is None:\n                b[0] = TILE_EMPTY\n            elif t == "LOCKED":\n                b[0] = TILE_LOCKED\n            elif isinstance(t, dict):\n                kind = t.get("kind")\n                if kind == "WEED":\n                    b[0] = TILE_WEED\n                elif kind == "PLANT":\n                    b[0] = TILE_PLANT\n                    b[1] = _CROP_CODE.get(t.get("crop"), 0)\n                    b[3] = _clamp_u8(t.get("yield_units", 0), 15)\n                    flags = 0\n                    if t.get("watered_today"):\n                        flags |= 1 << 0\n                    # fertilized_until_day >= current day means bonus active; we\n                    # don\'t have "today" here, so treat >=0 as "has fertilizer".\n                    if int(t.get("fertilized_until_day", -1)) >= 0:\n                        flags |= 1 << 1\n                    b[4] = flags\n                    b[5] = _clamp_u8(t.get("consecutive_unwatered", 0), 15)\n                elif kind in ("COOP", "PASTURE"):\n                    b[0] = TILE_COOP if kind == "COOP" else TILE_PASTURE\n                    b[2] = _ANIMAL_CODE.get(t.get("animal"), 0)\n                    b[3] = _clamp_u8(t.get("yield_units", 0), 15)\n                    flags = 0\n                    if t.get("fed_today"):\n                        flags |= 1 << 2\n                    if t.get("cared_today"):\n                        flags |= 1 << 3\n                    if t.get("fertilizer_available"):\n                        flags |= 1 << 4\n                    b[4] = flags\n                    lo = _clamp_u8(t.get("consecutive_unfed", 0), 15)\n                    hi = _clamp_u8(t.get("pending_care_bonus", 0), 15)\n                    b[5] = (hi << 4) | lo\n    # unit presence (farmer/hands) -> flag bits 5,6\n    fx, fy = farm.get("farmer", [0, 0])[:2]\n    if 0 <= fy < BOARD and 0 <= fx < BOARD:\n        out[fy, fx, 4] |= 1 << 5\n    for h in farm.get("hands", []) or []:\n        hx, hy = h[:2]\n        if 0 <= hy < BOARD and 0 <= hx < BOARD:\n            out[hy, hx, 4] |= 1 << 6\n    return out\n\n\n# Number of scalar features (kept in sync with _pack_scalars).\ndef _scalar_layout():\n    names = ["day", "hour", "step", "my_money", "opp_money", "money_diff",\n             "hires_today", "overage"]\n    names += [f"my_quad_{q}" for q in QUADS]\n    names += [f"opp_quad_{q}" for q in QUADS]\n    names += [f"mkt_inv_{p}" for p in PRODUCTS]\n    names += [f"mkt_price_{p}" for p in PRODUCTS]\n    names += [f"shed_{it}" for it in SHED_ITEMS]\n    names += [f"seed_{c}" for c in CROPS]\n    names += [f"shop_{s}" for s in SHOPS]\n    return names\n\nSCALAR_NAMES = _scalar_layout()\nN_SCALARS = len(SCALAR_NAMES)\n\n\ndef _pack_scalars(obs: dict, player: int) -> np.ndarray:\n    farms = obs.get("farms", [])\n    me = farms[player] if player < len(farms) else {}\n    opp = farms[1 - player] if (1 - player) < len(farms) else {}\n    market = obs.get("market", {}) or {}\n    inv = market.get("inventory", {}) or {}\n    prices = market.get("prices", {}) or {}\n    town = obs.get("town", {}) or {}\n    shops = town.get("unlocked_shops", []) or []\n    priv = obs.get("private", {}) or {}\n    shed = priv.get("shed", {}) or {}\n    seeds = priv.get("seeds", {}) or {}\n\n    my_money = float(me.get("money", 0.0))\n    opp_money = float(opp.get("money", 0.0))\n    v = []\n    v.append(obs.get("day", 0) / 30.0)\n    v.append(obs.get("hour", 0) / 24.0)\n    v.append(obs.get("step", 0) / 720.0)\n    v.append(my_money / 1e5)\n    v.append(opp_money / 1e5)\n    v.append((my_money - opp_money) / 1e5)\n    v.append(me.get("hires_today", 0) / 8.0)\n    v.append(float(obs.get("remainingOverageTime", 0.0)) / 60.0)\n    myq = set(me.get("unlocked_quadrants", []) or [])\n    for q in QUADS:\n        v.append(1.0 if q in myq else 0.0)\n    oppq = set(opp.get("unlocked_quadrants", []) or [])\n    for q in QUADS:\n        v.append(1.0 if q in oppq else 0.0)\n    for p in PRODUCTS:\n        v.append(float(inv.get(p, 10000)) / 1e4)\n    for p in PRODUCTS:\n        v.append(float(prices.get(p, BASE_PRICE[p])) / max(1.0, BASE_PRICE[p]))\n    for it in SHED_ITEMS:\n        v.append(float(shed.get(it, 0)) / 100.0)\n    for c in CROPS:\n        v.append(float(seeds.get(c, 0)) / 50.0)\n    counts = {s: 0 for s in SHOPS}\n    for s in shops:\n        key = str(s).upper().replace(" ", "_")\n        if key in counts:\n            counts[key] += 1\n    for s in SHOPS:\n        v.append(counts[s] / 8.0)\n    return np.asarray(v, dtype=np.float16)\n\n\ndef encode_observation(obs: dict, player: int | None = None):\n    """\n    Returns (grid_codes uint8 [2,10,10,TILE_BYTES], scalars float16 [N_SCALARS]).\n    grid[0] is the acting player\'s farm, grid[1] is the opponent\'s farm.\n    """\n    if player is None:\n        player = int(obs.get("player", 0))\n    farms = obs.get("farms", [])\n    me = farms[player] if player < len(farms) else {}\n    opp = farms[1 - player] if (1 - player) < len(farms) else {}\n    grid = np.stack([_pack_farm(me), _pack_farm(opp)], axis=0)\n    scal = _pack_scalars(obs, player)\n    return grid, scal\n\n\n# --------------------------------------------------------------------------- #\n#  Action encoding (for behavioral cloning labels).                           #\n# --------------------------------------------------------------------------- #\ndef _unit_op_label(op_list) -> int:\n    """Map a single unit op [\'OP\', ...args] to a UNIT_OPS index."""\n    if not op_list:\n        return UNIT_OP_INDEX["PASS"]\n    op = op_list[0]\n    if op in UNIT_OP_INDEX:            # simple op, no arg\n        return UNIT_OP_INDEX[op]\n    if op == "PLANT" and len(op_list) > 1:\n        return UNIT_OP_INDEX.get(f"PLANT_{op_list[1]}", UNIT_OP_INDEX["PASS"])\n    if op == "PICKUP" and len(op_list) > 1:\n        return UNIT_OP_INDEX.get(f"PICKUP_{op_list[1]}", UNIT_OP_INDEX["PASS"])\n    if op == "PLACE" and len(op_list) > 1:\n        return UNIT_OP_INDEX.get(f"PLACE_{op_list[1]}", UNIT_OP_INDEX["PASS"])\n    return UNIT_OP_INDEX["PASS"]\n\n\ndef encode_action(action: dict):\n    """\n    Turn one recorded action dict into supervised labels.\n\n    Returns dict with:\n      unit_ops   : int list  [farmer_label, hand1_label, ...]  (length 1+n_hands)\n      market_qty : int   [len(MARKET_QTY_FIELDS)]  bucket idx per field\n      market_bin : int   [2]  (hire, buy_land) in {0,1}\n    """\n    action = action or {}\n    farmer = action.get("farmer")\n    hands = action.get("hands", []) or []\n    unit_ops = [_unit_op_label(farmer)]\n    for h in hands:\n        unit_ops.append(_unit_op_label(h))\n\n    qty = np.zeros(len(MARKET_QTY_FIELDS), dtype=np.int64)\n    binf = np.zeros(len(MARKET_BIN_FIELDS), dtype=np.int64)\n    # aggregate the ordered market list into intent buckets\n    sell_acc = {p: 0 for p in PRODUCTS}\n    buyp_acc = {"WHEAT": 0, "FERTILIZER": 0}\n    seed_acc = {c: 0 for c in CROPS}\n    animal_acc = {a: 0 for a in ANIMALS}\n    for order in action.get("market", []) or []:\n        if not order:\n            continue\n        kind = order[0]\n        if kind == "SELL" and len(order) >= 3 and order[1] in sell_acc:\n            sell_acc[order[1]] += int(order[2])\n        elif kind == "BUY_PRODUCT" and len(order) >= 3 and order[1] in buyp_acc:\n            buyp_acc[order[1]] += int(order[2])\n        elif kind == "BUY_SEED" and len(order) >= 3 and order[1] in seed_acc:\n            seed_acc[order[1]] += int(order[2])\n        elif kind == "BUY_ANIMAL" and len(order) >= 3 and order[1] in animal_acc:\n            animal_acc[order[1]] += int(order[2])\n        elif kind == "HIRE":\n            binf[0] = 1\n        elif kind == "BUY_LAND":\n            binf[1] = 1\n    for p in PRODUCTS:\n        qty[MARKET_FIELD_INDEX[f"SELL_{p}"]] = qty_to_bucket(sell_acc[p])\n    qty[MARKET_FIELD_INDEX["BUYP_WHEAT"]] = qty_to_bucket(buyp_acc["WHEAT"])\n    qty[MARKET_FIELD_INDEX["BUYP_FERTILIZER"]] = qty_to_bucket(buyp_acc["FERTILIZER"])\n    for c in CROPS:\n        qty[MARKET_FIELD_INDEX[f"BUYSEED_{c}"]] = qty_to_bucket(seed_acc[c])\n    for a in ANIMALS:\n        qty[MARKET_FIELD_INDEX[f"BUYANIMAL_{a}"]] = qty_to_bucket(animal_acc[a])\n    return {"unit_ops": unit_ops, "market_qty": qty, "market_bin": binf}\n\n\n# --------------------------------------------------------------------------- #\n#  Legality masking (used at inference to zero out impossible unit ops).      #\n#  This keeps the deployed agent from emitting illegal no-ops the policy       #\n#  might otherwise be tempted to output on out-of-distribution states.        #\n# --------------------------------------------------------------------------- #\ndef legal_unit_op_mask(obs: dict, player: int, unit_xy) -> np.ndarray:\n    """\n    Return a float mask [N_UNIT_OPS] with 1.0 for legal ops, 0.0 otherwise,\n    for a unit standing at unit_xy on the acting player\'s farm.\n    Conservative: when unsure we allow the op (mask=1) rather than forbid it.\n    """\n    m = np.ones(N_UNIT_OPS, dtype=np.float32)\n    farms = obs.get("farms", [])\n    me = farms[player] if player < len(farms) else {}\n    tiles = me.get("tiles") or []\n    x, y = int(unit_xy[0]), int(unit_xy[1])\n    tile = None\n    if 0 <= y < len(tiles) and 0 <= x < len(tiles[y]):\n        tile = tiles[y][x]\n    is_locked = (tile == "LOCKED")\n    is_empty = (tile is None)\n    is_plant = isinstance(tile, dict) and tile.get("kind") == "PLANT"\n    is_weed = isinstance(tile, dict) and tile.get("kind") == "WEED"\n    is_struct = isinstance(tile, dict) and tile.get("kind") in ("COOP", "PASTURE")\n    has_animal = is_struct and tile.get("animal")\n\n    def forbid(name):\n        if name in UNIT_OP_INDEX:\n            m[UNIT_OP_INDEX[name]] = 0.0\n\n    # On locked tiles only movement / shed ops make sense.\n    if is_locked:\n        for op in UNIT_OPS:\n            if op not in ("PASS", "NORTH", "SOUTH", "EAST", "WEST",\n                          "DROP") and not op.startswith("PICKUP_"):\n                m[UNIT_OP_INDEX[op]] = 0.0\n        return m\n\n    if not is_plant:\n        for op in ("WATER", "HARVEST", "FERTILIZE"):\n            forbid(op)\n    if not is_struct:\n        for op in ("FEED", "COLLECT_FERTILIZER", "CARE", "BUILD_COOP",\n                   "BUILD_PASTURE"):\n            pass  # BUILD_* need empty tile; FEED/CARE need animal -> handled below\n    if not (is_struct and has_animal):\n        for op in ("FEED", "COLLECT_FERTILIZER", "CARE"):\n            forbid(op)\n    if not is_empty:\n        forbid("BUILD_COOP")\n        forbid("BUILD_PASTURE")\n        for name in _PLANT_OPS:\n            m[UNIT_OP_INDEX[name]] = 0.0\n    if not (is_plant or is_weed or (is_struct and not has_animal)):\n        forbid("DIG")\n    return m\n\n\n# --------------------------------------------------------------------------- #\n#  Action DECODING (inference): heads -> engine action dict.                   #\n# --------------------------------------------------------------------------- #\ndef decode_unit_op(label: int):\n    """UNIT_OPS index -> [\'OP\', ...args] as the engine expects."""\n    name = UNIT_OPS[int(label)]\n    if name.startswith("PLANT_"):\n        return ["PLANT", name[len("PLANT_"):]]\n    if name.startswith("PICKUP_"):\n        return ["PICKUP", name[len("PICKUP_"):], 1]\n    if name.startswith("PLACE_"):\n        return ["PLACE", name[len("PLACE_"):]]\n    return [name]\n\n\ndef assemble_market(qty_labels, bin_labels, max_orders: int = 10):\n    """\n    Turn market-head predictions into a legal ordered market list.\n    Priority order is chosen to be economically sensible: sells first (realise\n    cash), then restock buys, then structural HIRE / BUY_LAND.\n    """\n    orders = []\n    # SELLs\n    for p in PRODUCTS:\n        q = QTY_BUCKETS[int(qty_labels[MARKET_FIELD_INDEX[f"SELL_{p}"]])]\n        if q > 0:\n            orders.append(["SELL", p, q])\n    # BUY_PRODUCT\n    for item, key in (("WHEAT", "BUYP_WHEAT"), ("FERTILIZER", "BUYP_FERTILIZER")):\n        q = QTY_BUCKETS[int(qty_labels[MARKET_FIELD_INDEX[key]])]\n        if q > 0:\n            orders.append(["BUY_PRODUCT", item, q])\n    # BUY_SEED\n    for c in CROPS:\n        q = QTY_BUCKETS[int(qty_labels[MARKET_FIELD_INDEX[f"BUYSEED_{c}"]])]\n        if q > 0:\n            orders.append(["BUY_SEED", c, q])\n    # BUY_ANIMAL\n    for a in ANIMALS:\n        q = QTY_BUCKETS[int(qty_labels[MARKET_FIELD_INDEX[f"BUYANIMAL_{a}"]])]\n        if q > 0:\n            orders.append(["BUY_ANIMAL", a, q])\n    if int(bin_labels[0]) == 1:\n        orders.append(["HIRE"])\n    if int(bin_labels[1]) == 1:\n        orders.append(["BUY_LAND"])\n    return orders[:max_orders]\n\n\n# --------------------------------------------------------------------------- #\n#  Convenience: expand packed grid codes to float one-hot channels.           #\n#  Called on-device in kagri.datasets; put here so train + infer share it.    #\n# --------------------------------------------------------------------------- #\n# channel layout produced by expand_grid():\nGRID_CHANNELS = (\n    6            # tile type one-hot\n    + len(CROPS)      # crop one-hot\n    + len(ANIMALS)    # animal one-hot\n    + 1               # yield_units / 15\n    + 7               # 7 flag bits\n    + 2               # consecutive / pending_care (normalised)\n)  # per farm\n\ndef expand_grid_numpy(grid_codes: np.ndarray) -> np.ndarray:\n    """\n    grid_codes : uint8 [2,10,10,TILE_BYTES]  ->  float32 [2*GRID_CHANNELS,10,10]\n    Reference numpy implementation (the torch version in datasets.py mirrors it).\n    """\n    farms = []\n    for f in range(grid_codes.shape[0]):\n        g = grid_codes[f].astype(np.float32)  # [10,10,6]\n        tt = g[..., 0].astype(np.int64)\n        crop = g[..., 1].astype(np.int64)\n        animal = g[..., 2].astype(np.int64)\n        chans = []\n        for k in range(6):\n            chans.append((tt == k).astype(np.float32))\n        for k in range(1, len(CROPS) + 1):\n            chans.append((crop == k).astype(np.float32))\n        for k in range(1, len(ANIMALS) + 1):\n            chans.append((animal == k).astype(np.float32))\n        chans.append(g[..., 3] / 15.0)\n        flags = g[..., 4].astype(np.int64)\n        for bit in range(7):\n            chans.append(((flags >> bit) & 1).astype(np.float32))\n        misc = g[..., 5].astype(np.int64)\n        chans.append(((misc & 0x0F).astype(np.float32)) / 15.0)\n        chans.append(((misc >> 4).astype(np.float32)) / 15.0)\n        farms.append(np.stack(chans, axis=0))  # [C,10,10]\n    return np.concatenate(farms, axis=0)\n\n\nif __name__ == "__main__":\n    # tiny self-test with a synthetic observation\n    obs = {\n        "player": 0, "day": 3, "hour": 5, "step": 77,\n        "remainingOverageTime": 55,\n        "farms": [\n            {"money": 1234, "hires_today": 2, "farmer": [4, 4],\n             "hands": [[5, 4]], "unlocked_quadrants": ["NW", "NE"],\n             "tiles": [[None] * 10 for _ in range(10)]},\n            {"money": 999, "farmer": [4, 4], "hands": [],\n             "unlocked_quadrants": ["NW"],\n             "tiles": [["LOCKED"] * 10 for _ in range(10)]},\n        ],\n        "market": {"inventory": {p: 9000 for p in PRODUCTS},\n                   "prices": {p: BASE_PRICE[p] for p in PRODUCTS}},\n        "town": {"unlocked_shops": ["BAKERY", "YARN_STORE"]},\n        "private": {"shed": {"WHEAT": 5}, "seeds": {"WHEAT": 2},\n                    "inventories": [{}]},\n    }\n    obs["farms"][0]["tiles"][3][3] = {\n        "kind": "PLANT", "crop": "MELON", "planted_day": 1, "watered_today": True,\n        "consecutive_unwatered": 0, "yield_units": 3, "max_lifespan_step": 300,\n        "fertilized_until_day": 5}\n    g, s = encode_observation(obs)\n    print("grid", g.shape, g.dtype, "scalars", s.shape, s.dtype)\n    exp = expand_grid_numpy(g)\n    print("expanded grid", exp.shape, "channels/farm", GRID_CHANNELS)\n    act = {"farmer": ["PLANT", "MELON"], "hands": [["WATER"]],\n           "market": [["SELL", "WHEAT", 5], ["BUY_SEED", "MELON", 2], ["HIRE"]]}\n    lab = encode_action(act)\n    print("unit_ops", lab["unit_ops"], "-> decode0", decode_unit_op(lab["unit_ops"][0]))\n    print("market_qty nonzero", np.nonzero(lab["market_qty"])[0], "bin", lab["market_bin"])\n    print("N_UNIT_OPS", N_UNIT_OPS, "N_SCALARS", N_SCALARS,\n          "market fields", len(MARKET_QTY_FIELDS))\n    print("assemble", assemble_market(lab["market_qty"], lab["market_bin"]))\n    print("SELF-TEST OK")\n', 'kagri/models.py': '"""Models used by the Kaggriculture offline-learning pipeline.\n\nThe BC architecture is deliberately conservative but fixes one structural\nweakness in the previous version: global average pooling discarded coarse board\nlocation.  A 2x2 adaptive spatial pool retains coarse spatial layout while\nkeeping the model small enough for T4/P100 free-tier training.\n"""\nfrom __future__ import annotations\nimport torch\nimport torch.nn as nn\nfrom .datasets import TOTAL_GRID_CHANNELS\nfrom .encoding import N_SCALARS, N_UNIT_OPS, MARKET_QTY_FIELDS, MARKET_BIN_FIELDS, N_QTY\n\n\nclass GridScalarTrunk(nn.Module):\n    def __init__(self, dim=256, dropout=0.1, spatial_pool=2):\n        super().__init__()\n        c=TOTAL_GRID_CHANNELS\n        self.conv=nn.Sequential(\n            nn.Conv2d(c,64,3,padding=1), nn.GroupNorm(8,64), nn.SiLU(),\n            nn.Conv2d(64,96,3,padding=1), nn.GroupNorm(8,96), nn.SiLU(),\n            nn.Conv2d(96,128,3,padding=1), nn.GroupNorm(8,128), nn.SiLU(),\n        )\n        self.pool=nn.AdaptiveAvgPool2d((spatial_pool,spatial_pool))\n        spatial_dim=128*spatial_pool*spatial_pool\n        self.scalar_mlp=nn.Sequential(\n            nn.Linear(N_SCALARS,128), nn.SiLU(), nn.Dropout(dropout),\n            nn.Linear(128,128), nn.SiLU(),\n        )\n        self.head=nn.Sequential(\n            nn.Linear(spatial_dim+128,dim), nn.SiLU(), nn.Dropout(dropout),\n            nn.Linear(dim,dim), nn.SiLU(),\n        )\n        self.dim=dim\n\n    def forward(self, grid, scalar):\n        g=self.pool(self.conv(grid)).flatten(1)\n        s=self.scalar_mlp(scalar)\n        return self.head(torch.cat([g,s],1))\n\n\nclass BCPolicy(nn.Module):\n    def __init__(self, dim=256, dropout=0.1, spatial_pool=2):\n        super().__init__()\n        self.trunk=GridScalarTrunk(dim,dropout,spatial_pool)\n        self.unit_head=nn.Sequential(\n            nn.Linear(dim+2,dim),nn.SiLU(),nn.Dropout(dropout),nn.Linear(dim,N_UNIT_OPS)\n        )\n        self.mkt_qty_head=nn.Linear(dim,len(MARKET_QTY_FIELDS)*N_QTY)\n        self.mkt_bin_head=nn.Linear(dim,len(MARKET_BIN_FIELDS)*2)\n        self.value_head=nn.Sequential(nn.Linear(dim,64),nn.SiLU(),nn.Linear(64,1))\n\n    def forward(self,grid,scalar,unit_feat):\n        z=self.trunk(grid,scalar)\n        unit_logits=self.unit_head(torch.cat([z,unit_feat],1))\n        qty=self.mkt_qty_head(z).view(-1,len(MARKET_QTY_FIELDS),N_QTY)\n        binl=self.mkt_bin_head(z).view(-1,len(MARKET_BIN_FIELDS),2)\n        value=self.value_head(z).squeeze(-1)\n        return unit_logits,qty,binl,value\n\n    @torch.no_grad()\n    def embed(self,grid,scalar): return self.trunk(grid,scalar)\n\n\nclass IQLNets(nn.Module):\n    def __init__(self,dim=256,dropout=0.1,spatial_pool=2):\n        super().__init__()\n        self.q_trunk=GridScalarTrunk(dim,dropout,spatial_pool)\n        self.q_head=nn.Sequential(nn.Linear(dim+2,dim),nn.SiLU(),nn.Linear(dim,N_UNIT_OPS))\n        self.v_trunk=GridScalarTrunk(dim,dropout,spatial_pool)\n        self.v_head=nn.Sequential(nn.Linear(dim,dim),nn.SiLU(),nn.Linear(dim,1))\n        self.policy=BCPolicy(dim,dropout,spatial_pool)\n    def q(self,grid,scalar,unit_feat):\n        z=self.q_trunk(grid,scalar); return self.q_head(torch.cat([z,unit_feat],1))\n    def v(self,grid,scalar): return self.v_head(self.v_trunk(grid,scalar)).squeeze(-1)\n\n\nclass DecisionTransformer(nn.Module):\n    def __init__(self,dim=256,n_layers=4,n_heads=8,context=40,dropout=0.1,max_timestep=720,spatial_pool=2):\n        super().__init__(); self.dim=dim; self.context=context\n        self.trunk=GridScalarTrunk(dim,dropout,spatial_pool)\n        self.rtg_embed=nn.Linear(1,dim); self.act_embed=nn.Embedding(N_UNIT_OPS,dim)\n        self.time_embed=nn.Embedding(max_timestep+1,dim)\n        layer=nn.TransformerEncoderLayer(d_model=dim,nhead=n_heads,dim_feedforward=4*dim,\n                                         dropout=dropout,activation="gelu",batch_first=True)\n        self.transformer=nn.TransformerEncoder(layer,n_layers)\n        self.ln=nn.LayerNorm(dim); self.act_pred=nn.Linear(dim,N_UNIT_OPS)\n    def forward(self,grid,scalar,rtg,action,timestep):\n        B,W=grid.shape[:2]\n        st=self.trunk(grid.reshape(B*W,*grid.shape[2:]),scalar.reshape(B*W,scalar.shape[2])).reshape(B,W,-1)\n        te=self.time_embed(timestep.clamp(max=self.time_embed.num_embeddings-1))\n        rt=self.rtg_embed(rtg.unsqueeze(-1))+te; st=st+te; at=self.act_embed(action)+te\n        tokens=torch.stack([rt,st,at],2).reshape(B,3*W,self.dim)\n        L=tokens.shape[1]; causal=torch.triu(torch.ones(L,L,device=tokens.device),1).bool()\n        h=self.transformer(self.ln(tokens),mask=causal)\n        return self.act_pred(h[:,1::3])\n\n\ndef count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)\n', 'kagri/train.py': '"""Unattended-safe BC training for Kaggle free GPUs.\n\nDesign decisions are tied to observed evidence:\n- no torch.compile (the previous run produced Dynamo/thread recompilation warnings)\n- no DataParallel; use DDP when launched with torchrun (official PyTorch guidance)\n- AMP FP16 for T4 Tensor Cores\n- shard-local batch sampling to reduce compressed-NPZ cache churn\n- pinned memory + non_blocking H2D copies\n- correct unit-example counting\n- checkpoint/resume with atomic writes\n"""\nfrom __future__ import annotations\nimport os, math, time, random\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader\n\nfrom .datasets import UnitTransitionDataset, ShardBatchSampler, collate_unit, expand_grid_torch\nfrom .models import BCPolicy, count_params\nfrom .preflight import inspect_processed, print_environment\n\n\ndef _setup_cuda():\n    if not torch.cuda.is_available():\n        return\n    # T4 is Turing, so TF32 is not an optimization we rely on.  FP16 AMP is.\n    torch.backends.cudnn.benchmark = True\n\n\ndef _dist_info():\n    return (\n        int(os.environ.get("RANK", "0")),\n        int(os.environ.get("LOCAL_RANK", "0")),\n        int(os.environ.get("WORLD_SIZE", "1")),\n    )\n\n\ndef _init_ddp(use_ddp=True):\n    rank, local_rank, world = _dist_info()\n    if use_ddp and world > 1:\n        torch.cuda.set_device(local_rank)\n        dist.init_process_group(backend="nccl")\n        return rank, local_rank, world, True\n    if torch.cuda.is_available():\n        torch.cuda.set_device(0)\n    return rank, 0, 1, False\n\n\ndef _barrier(ddp):\n    if ddp: dist.barrier()\n\n\ndef _is_main(rank): return rank == 0\n\n\ndef _reduce_sum(value, device, ddp):\n    t = torch.tensor(float(value), device=device, dtype=torch.float64)\n    if ddp: dist.all_reduce(t, op=dist.ReduceOp.SUM)\n    return float(t.item())\n\n\ndef _cosine_warmup(opt, warmup, total):\n    def f(step):\n        if step < warmup: return step / max(1, warmup)\n        p=(step-warmup)/max(1,total-warmup)\n        return 0.5*(1+math.cos(math.pi*p))\n    return torch.optim.lr_scheduler.LambdaLR(opt, f)\n\n\ndef _make_loader(ds, batch_size, workers, prefetch, shuffle, seed, rank, world):\n    sampler=ShardBatchSampler(ds.shards,batch_size,shuffle,seed,rank,world,True)\n    kwargs=dict(batch_sampler=sampler,collate_fn=collate_unit,pin_memory=True,\n                num_workers=workers,persistent_workers=workers>0)\n    if workers>0: kwargs["prefetch_factor"]=prefetch\n    # Do not use DataLoader shuffle/sampler: batch_sampler owns ordering.\n    return DataLoader(ds,**kwargs), sampler\n\n\ndef _atomic_torch_save(obj, path):\n    tmp=path+".tmp"\n    torch.save(obj,tmp)\n    os.replace(tmp,path)\n\n\ndef _save_rng():\n    state={"python":random.getstate(),"numpy":np.random.get_state(),"torch":torch.get_rng_state()}\n    if torch.cuda.is_available(): state["cuda"]=torch.cuda.get_rng_state_all()\n    return state\n\n\ndef _load_rng(state):\n    if not state: return\n    random.setstate(state["python"]); np.random.set_state(state["numpy"]); torch.set_rng_state(state["torch"])\n    if torch.cuda.is_available() and "cuda" in state: torch.cuda.set_rng_state_all(state["cuda"])\n\n\ndef _unwrap(model): return model.module if isinstance(model,DDP) else model\n\n\ndef _move_batch(batch, device):\n    # Small integer targets stay compact on CPU and are cast on GPU.\n    grid=batch["grid_codes"].to(device,non_blocking=True)\n    scalar=batch["scalar"].to(device,non_blocking=True)\n    unit_feat=batch["unit_feat"].to(device,non_blocking=True)\n    op=batch["unit_op"].to(device,non_blocking=True).long()\n    mqty=batch["market_qty"].to(device,non_blocking=True).long()\n    mbin=batch["market_bin"].to(device,non_blocking=True).long()\n    fr=batch["final_reward"].to(device,non_blocking=True)\n    win=batch["win"].to(device,non_blocking=True)\n    tw=batch["turn_weight"].to(device,non_blocking=True)\n    return grid,scalar,unit_feat,op,mqty,mbin,fr,win,tw\n\n\ndef _eval_bc(model, loader, sampler, device, ls, max_examples, ddp):\n    model.eval(); total_loss=0.0; total_n=0.0; correct=0.0; seen=0.0\n    local_limit=max(1,int(math.ceil(max_examples/max(1,dist.get_world_size() if ddp else 1))))\n    with torch.no_grad():\n        for batch in loader:\n            grid,scalar,uf,op,mqty,mbin,fr,win,tw=_move_batch(batch,device)\n            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=device.type=="cuda"):\n                g=expand_grid_torch(grid)\n                ulog,qty,binl,value=model(g,scalar,uf)\n                lu=F.cross_entropy(ulog,op,label_smoothing=ls)\n            bs=op.numel(); total_loss += float(lu.item())*bs; total_n += bs\n            correct += float((ulog.argmax(-1)==op).sum().item()); seen += bs\n            if seen>=local_limit: break\n    total_loss=_reduce_sum(total_loss,device,ddp); total_n=_reduce_sum(total_n,device,ddp)\n    correct=_reduce_sum(correct,device,ddp); seen=_reduce_sum(seen,device,ddp)\n    return total_loss/max(1,total_n), correct/max(1,seen)\n\n\ndef train_bc(cfg):\n    _setup_cuda()\n    rank,local_rank,world,ddp=_init_ddp(cfg.get("use_ddp",True))\n    device=torch.device("cuda",local_rank) if torch.cuda.is_available() else torch.device("cpu")\n    if _is_main(rank): print_environment()\n\n    out_dir=cfg["out_dir"]; work=cfg["work_dir"]; os.makedirs(work,exist_ok=True)\n    if cfg.get("preflight", True):\n        # Only rank 0 performs the expensive shard-by-shard integrity scan.\n        # Other ranks wait for the result before constructing datasets.\n        info = None\n        if _is_main(rank):\n            info = inspect_processed(out_dir)\n            print(\n                f"Processed shards={info[\'shards\']:,} "\n                f"turn_rows={info[\'turn_rows\']:,} "\n                f"unit_examples={info[\'unit_examples\']:,}",\n                flush=True,\n            )\n            print("The BC dataset length is unit_examples, not turn_rows.", flush=True)\n        _barrier(ddp)\n\n    tr=UnitTransitionDataset(out_dir,"train",cfg.get("val_frac",0.05),cfg.get("seed",0))\n    va=UnitTransitionDataset(out_dir,"val",cfg.get("val_frac",0.05),cfg.get("seed",0))\n    if _is_main(rank): print(f"BC train examples={len(tr):,} val={len(va):,}")\n\n    bs=cfg.get("batch_size",2048); workers=cfg.get("workers",4); prefetch=cfg.get("prefetch",2); seed=cfg.get("seed",0)\n    tl,ts=_make_loader(tr,bs,workers,prefetch,True,seed,rank,world)\n    vl,vs=_make_loader(va,bs,workers,prefetch,False,seed,rank,world)\n\n    base=BCPolicy(cfg.get("dim",256),cfg.get("dropout",0.1),cfg.get("spatial_pool",2)).to(device)\n    if ddp:\n        model=DDP(base,device_ids=[local_rank],output_device=local_rank,gradient_as_bucket_view=True)\n    else:\n        model=base\n        if torch.cuda.device_count()>1 and _is_main(rank):\n            print("Not using DataParallel. Launch with torchrun for 2-GPU DDP.")\n    if _is_main(rank):\n        print(f"Training mode: {\'DDP\' if ddp else \'single-GPU\'}")\n        print("torch.compile disabled (intentional stable path)")\n        print("params:",count_params(base))\n\n    opt_kwargs=dict(lr=cfg.get("lr",3e-4),weight_decay=cfg.get("weight_decay",1e-4))\n    if device.type=="cuda": opt_kwargs["fused"]=True\n    opt=torch.optim.AdamW(model.parameters(),**opt_kwargs)\n    total_steps=cfg.get("epochs",6)*len(tl)\n    sched=_cosine_warmup(opt,cfg.get("warmup",500),total_steps)\n    scaler=torch.amp.GradScaler("cuda",enabled=device.type=="cuda")\n    ls=cfg.get("label_smoothing",0.05); rw=cfg.get("return_weighting",True)\n\n    start_epoch=0; step=0; best=float("inf"); bad=0\n    last_path=os.path.join(work,"bc_last.pt"); best_path=os.path.join(work,"bc_best.pt")\n    if os.path.exists(last_path):\n        _barrier(ddp)\n        ckpt=torch.load(last_path,map_location=device,weights_only=False)\n        base.load_state_dict(ckpt["model"])\n        if "opt" in ckpt: opt.load_state_dict(ckpt["opt"])\n        if "sched" in ckpt: sched.load_state_dict(ckpt["sched"])\n        if "scaler" in ckpt: scaler.load_state_dict(ckpt["scaler"])\n        start_epoch=int(ckpt.get("epoch",-1))+1; step=int(ckpt.get("step",0)); best=float(ckpt.get("best",float("inf"))); bad=int(ckpt.get("bad",0))\n        if "rng" in ckpt: _load_rng(ckpt["rng"])\n        if _is_main(rank): print(f"Resumed checkpoint: epoch={start_epoch} step={step} best={best:.5f}")\n\n    for ep in range(start_epoch,cfg.get("epochs",6)):\n        ts.set_epoch(ep); vs.set_epoch(ep)\n        model.train(); t0=time.time(); running=0.0; running_n=0\n        for batch in tl:\n            grid,scalar,uf,op,mqty,mbin,fr,win,tw=_move_batch(batch,device)\n            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=device.type=="cuda"):\n                g=expand_grid_torch(grid)\n                ulog,qty,binl,value=model(g,scalar,uf)\n                if rw: w=0.5+win\n                else: w=torch.ones_like(win)\n                unit_per=F.cross_entropy(ulog,op,reduction="none",label_smoothing=ls)\n                lu=(unit_per*w).mean()\n                qty_per=F.cross_entropy(qty.reshape(-1,qty.shape[-1]),mqty.reshape(-1),reduction="none",label_smoothing=ls).reshape(qty.shape[0],qty.shape[1]).mean(1)\n                bin_per=F.cross_entropy(binl.reshape(-1,binl.shape[-1]),mbin.reshape(-1),reduction="none",label_smoothing=ls).reshape(binl.shape[0],binl.shape[1]).mean(1)\n                lm=((qty_per+bin_per)*tw.float()).sum()/tw.float().sum().clamp_min(1)\n                lv=(F.smooth_l1_loss(value,fr/1e5,reduction="none")*tw.float()).sum()/tw.float().sum().clamp_min(1)\n                loss=lu+cfg.get("market_w",0.5)*lm+cfg.get("value_w",0.1)*lv\n            opt.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            nn.utils.clip_grad_norm_(model.parameters(),cfg.get("clip",1.0))\n            scaler.step(opt); scaler.update(); sched.step()\n            running += float(loss.item()); running_n += 1; step += 1\n            if step % cfg.get("log_every",200)==0 and _is_main(rank):\n                elapsed=time.time()-t0; samples=step and cfg.get("log_every",200)*bs*world/max(elapsed,1e-9)\n                print(f"ep{ep} step{step} loss={running/max(1,running_n):.4f} lr={sched.get_last_lr()[0]:.2e} "\n                      f"({elapsed/60:.1f}m, {samples:.0f} samples/s)",flush=True)\n                running=0.0; running_n=0\n\n        _barrier(ddp)\n        vloss,vacc=_eval_bc(model,vl,vs,device,ls,cfg.get("val_max_examples",500_000),ddp)\n        if _is_main(rank):\n            print(f"[VAL] ep{ep} loss={vloss:.5f} unit_acc={vacc:.4f}")\n            improved = vloss < best\n            if improved:\n                best=vloss; bad=0\n            else:\n                bad += 1\n            payload={"model":base.state_dict(),"opt":opt.state_dict(),"sched":sched.state_dict(),"scaler":scaler.state_dict(),\n                     "cfg":cfg,"epoch":ep,"step":step,"val_loss":vloss,"val_acc":vacc,"best":best,"bad":bad,"rng":_save_rng()}\n            _atomic_torch_save(payload,last_path)\n            if improved:\n                _atomic_torch_save(payload,best_path); print(f"  saved new best: {best:.5f}")\n            else:\n                print(f"  no improvement ({bad}/{cfg.get(\'patience\',3)})")\n        _barrier(ddp)\n        if ddp:\n            # All ranks need the updated early-stopping state.\n            t=torch.tensor([best,bad],device=device,dtype=torch.float64)\n            dist.broadcast(t,src=0); best=float(t[0]); bad=int(t[1].item())\n        if bad>=cfg.get("patience",3):\n            if _is_main(rank): print("Early stopping.")\n            break\n\n    _barrier(ddp)\n    if ddp: dist.destroy_process_group()\n    return best_path\n', 'kagri/preprocess.py': '"""Streaming replay preprocessing with a correct index manifest."""\nfrom __future__ import annotations\nimport os, json, glob, time, traceback\nfrom pathlib import Path\nfrom multiprocessing import Pool, cpu_count\nimport numpy as np\nfrom .encoding import encode_observation, encode_action\n\nMAX_UNITS_PER_TURN = 12\n\ntry:\n    import orjson\n    def _read_json(path):\n        with open(path, "rb") as f:\n            return orjson.loads(f.read())\nexcept Exception:\n    def _read_json(path):\n        with open(path, "r") as f:\n            return json.load(f)\n\n\ndef _final_rewards(replay, n_players):\n    steps = replay.get("steps", [])\n    if not steps:\n        return [0.0] * n_players\n    last = steps[-1]\n    out = []\n    for p in range(n_players):\n        r = last[p].get("reward") if p < len(last) else None\n        if r is None and p < len(last):\n            obs = last[p].get("observation", {}) or {}\n            farms = obs.get("farms", [])\n            if p < len(farms): r = farms[p].get("money")\n        out.append(float(r) if r is not None else 0.0)\n    return out\n\n\ndef _player_money(obs, player):\n    farms = (obs or {}).get("farms", [])\n    return float(farms[player].get("money", 0.0)) if player < len(farms) else 0.0\n\n\ndef process_replay(path, stride=1):\n    try: replay = _read_json(path)\n    except Exception: return None\n    steps = replay.get("steps", [])\n    if len(steps) < 2: return None\n    n_players = len(steps[0])\n    finals = _final_rewards(replay, n_players)\n    mx = max(finals)\n    win = [1.0 if finals[p] == mx and finals.count(mx) == 1 else (0.5 if finals[p] == mx else 0.0)\n           for p in range(n_players)]\n    grids=[]; scalars=[]; unit_ops_flat=[]; unit_counts=[]; mkt_qty=[]; mkt_bin=[]\n    rewards=[]; ep_ids=[]; players=[]; stepids=[]\n    stem = Path(path).stem\n    episode_id = int(stem.split("_")[0]) if stem[:1].isdigit() else abs(hash(stem)) % 10**9\n    for t in range(0, len(steps), stride):\n        entry = steps[t]\n        for p in range(min(n_players, len(entry))):\n            rec = entry[p]; obs = rec.get("observation"); act = rec.get("action")\n            if obs is None or act is None: continue\n            if "player" not in obs:\n                obs = dict(obs); obs["player"] = p\n            try: g, s = encode_observation(obs, p); lab = encode_action(act)\n            except Exception: continue\n            uops = lab["unit_ops"][:MAX_UNITS_PER_TURN]\n            if not uops: continue\n            r = 0.0\n            if t + 1 < len(steps) and p < len(steps[t + 1]):\n                nxt = steps[t + 1][p].get("observation", {})\n                r = _player_money(nxt, p) - _player_money(obs, p)\n            grids.append(g); scalars.append(s); unit_ops_flat.extend(uops); unit_counts.append(len(uops))\n            mkt_qty.append(lab["market_qty"]); mkt_bin.append(lab["market_bin"])\n            rewards.append(r); ep_ids.append(episode_id); players.append(p); stepids.append(t)\n    if not grids: return None\n    return {\n        "grids": np.asarray(grids, np.uint8), "scalars": np.asarray(scalars, np.float16),\n        "unit_ops": np.asarray(unit_ops_flat, np.int16), "unit_counts": np.asarray(unit_counts, np.int16),\n        "market_qty": np.asarray(mkt_qty, np.int8), "market_bin": np.asarray(mkt_bin, np.int8),\n        "rewards": np.asarray(rewards, np.float32), "episode_id": np.asarray(ep_ids, np.int64),\n        "player": np.asarray(players, np.int8), "step": np.asarray(stepids, np.int16),\n        "final_reward": np.asarray([finals[p] for p in players], np.float32),\n        "win": np.asarray([win[p] for p in players], np.float16),\n    }\n\n\ndef _worker(args):\n    path, stride, out_dir, compressed = args\n    name = Path(path).stem; out_path = os.path.join(out_dir, f"shard_{name}.npz")\n    if os.path.exists(out_path): return name, "skip", 0\n    try:\n        data = process_replay(path, stride)\n        if data is None: return name, "empty", 0\n        saver = np.savez_compressed if compressed else np.savez\n        saver(out_path, **data)\n        return name, "ok", int(data["grids"].shape[0])\n    except Exception:\n        return name, "err:" + traceback.format_exc().splitlines()[-1], 0\n\n\ndef gather_replays(dataset_dirs):\n    files=[]\n    for d in dataset_dirs:\n        p=Path(d)\n        if p.exists(): files.extend(sorted(str(f) for f in p.rglob("*.json")))\n    return files\n\n\ndef write_index(out_dir):\n    shards = sorted(glob.glob(os.path.join(out_dir, "shard_*.npz")))\n    turn_counts=[]; unit_counts=[]\n    for sp in shards:\n        try:\n            with np.load(sp, allow_pickle=False) as z:\n                tc = int(z["unit_counts"].shape[0])\n                uc = int(z["unit_counts"].astype(np.int64).sum())\n        except Exception:\n            tc=uc=0\n        turn_counts.append(tc); unit_counts.append(uc)\n    np.savez(\n        os.path.join(out_dir, "index.npz"),\n        shards=np.asarray([os.path.basename(s) for s in shards]),\n        counts=np.asarray(turn_counts, np.int64),\n        unit_counts=np.asarray(unit_counts, np.int64),\n    )\n    print(f"Index written: {len(shards):,} shards, {sum(turn_counts):,} turn rows, {sum(unit_counts):,} unit examples.")\n\n\ndef run_preprocess(dataset_dirs, out_dir, stride=1, max_episodes=None, workers=None, compressed=True):\n    os.makedirs(out_dir, exist_ok=True)\n    done_path=os.path.join(out_dir,"_done.txt")\n    done=set(open(done_path).read().split()) if os.path.exists(done_path) else set()\n    files=[f for f in gather_replays(dataset_dirs) if Path(f).stem not in done]\n    if max_episodes: files=files[:max_episodes]\n    if not files:\n        write_index(out_dir); print("Nothing to do — index refreshed."); return\n    workers=workers or max(1,cpu_count())\n    print(f"Preprocessing {len(files):,} replays with {workers} workers (stride={stride})")\n    args=[(f,stride,out_dir,compressed) for f in files]; t0=time.time(); n_rows=0\n    with Pool(workers) as pool, open(done_path,"a") as dh:\n        for i,(name,status,rows) in enumerate(pool.imap_unordered(_worker,args),1):\n            if status in ("ok","skip","empty"):\n                dh.write(name+"\\n"); dh.flush()\n            n_rows += rows\n            if i%25==0 or i==len(files):\n                el=time.time()-t0\n                print(f"[{i:>5}/{len(files)}] {name:>18} {status:<6} rows={rows:<6} total_rows={n_rows:,} elapsed={el/60:.1f}m rate={i/max(el,1):.1f} files/s")\n    write_index(out_dir)\n', 'kagri/meta_mining.py': '"""\nkagri.meta_mining\n=================\nData-mining over the preprocessed shards to surface *what the winners actually\ndo* — the practical route to "a meta that hasn\'t been found yet".\n\nIt does three things, all purely descriptive (numpy only, runs anywhere):\n\n  1. RANK episodes by final bank balance.\n  2. FINGERPRINT each (episode, player): per-strategy-lever aggregates\n     (crops planted, animals bought, land buys, hiring intensity, product sales\n     mix, timing of first land/animal), computed from the stored action labels.\n  3. CORRELATE every lever with final balance, and contrast the TOP-k vs the\n     BOTTOM-k cohorts, printing a human-readable "meta report" + a CSV.\n\nThis tells you, empirically and per the real replays, which strategic choices\ncovary with high scores — the hypotheses your trained agent (BC/IQL/DT) then\nexecutes and extrapolates beyond.\n"""\nfrom __future__ import annotations\nimport os, glob, csv\nimport numpy as np\n\nfrom .encoding import (UNIT_OPS, UNIT_OP_INDEX, CROPS, ANIMALS, PRODUCTS,\n                       MARKET_QTY_FIELDS, MARKET_FIELD_INDEX, QTY_BUCKETS)\n\n_PLANT_LABELS = {c: UNIT_OP_INDEX[f"PLANT_{c}"] for c in CROPS}\n_BUILD_COOP = UNIT_OP_INDEX["BUILD_COOP"]\n_BUILD_PASTURE = UNIT_OP_INDEX["BUILD_PASTURE"]\n_HARVEST = UNIT_OP_INDEX["HARVEST"]\n_FERTILIZE = UNIT_OP_INDEX["FERTILIZE"]\n\n\ndef _iter_shards(out_dir):\n    for sp in sorted(glob.glob(os.path.join(out_dir, "shard_*.npz"))):\n        with np.load(sp) as z:\n            yield {k: z[k] for k in z.files}\n\n\ndef fingerprint(out_dir, max_shards=None):\n    """Return list of per-(episode,player) dicts with strategic aggregates."""\n    rows = []\n    for si, d in enumerate(_iter_shards(out_dir)):\n        if max_shards and si >= max_shards:\n            break\n        ep = d["episode_id"]; pl = d["player"]; st = d["step"]\n        final_r = d["final_reward"]; win = d["win"]\n        counts = d["unit_counts"].astype(np.int64)\n        offs = np.concatenate([[0], np.cumsum(counts)])\n        uops = d["unit_ops"]\n        mqty = d["market_qty"]; mbin = d["market_bin"]\n        # group turn-rows by (ep,player)\n        groups = {}\n        for r in range(len(ep)):\n            groups.setdefault((int(ep[r]), int(pl[r])), []).append(r)\n        for (e, p), rr in groups.items():\n            rr = sorted(rr, key=lambda x: int(st[x]))\n            f = {"episode": e, "player": p,\n                 "final_balance": float(final_r[rr[0]]),\n                 "win": float(win[rr[0]])}\n            # unit-op tallies\n            plant = {c: 0 for c in CROPS}\n            n_harvest = n_fert = n_coop = n_pasture = 0\n            for r in rr:\n                for u in range(offs[r], offs[r + 1]):\n                    op = int(uops[u])\n                    for c, lab in _PLANT_LABELS.items():\n                        if op == lab:\n                            plant[c] += 1\n                    if op == _HARVEST: n_harvest += 1\n                    elif op == _FERTILIZE: n_fert += 1\n                    elif op == _BUILD_COOP: n_coop += 1\n                    elif op == _BUILD_PASTURE: n_pasture += 1\n            for c in CROPS:\n                f[f"plant_{c}"] = plant[c]\n            f["harvests"] = n_harvest\n            f["fertilizes"] = n_fert\n            f["coops_built"] = n_coop\n            f["pastures_built"] = n_pasture\n            # market intents\n            sells = {p2: 0 for p2 in PRODUCTS}\n            buy_land = 0; hires = 0; buy_animal = {a: 0 for a in ANIMALS}\n            first_land_step = -1\n            for r in rr:\n                for p2 in PRODUCTS:\n                    sells[p2] += QTY_BUCKETS[int(mqty[r][MARKET_FIELD_INDEX[f"SELL_{p2}"]])]\n                for a in ANIMALS:\n                    buy_animal[a] += QTY_BUCKETS[int(mqty[r][MARKET_FIELD_INDEX[f"BUYANIMAL_{a}"]])]\n                if int(mbin[r][0]) == 1: hires += 1\n                if int(mbin[r][1]) == 1:\n                    buy_land += 1\n                    if first_land_step < 0:\n                        first_land_step = int(st[r])\n            for p2 in PRODUCTS:\n                f[f"sell_{p2}"] = sells[p2]\n            for a in ANIMALS:\n                f[f"buy_{a}"] = buy_animal[a]\n            f["land_buys"] = buy_land\n            f["first_land_step"] = first_land_step\n            f["hire_turns"] = hires\n            rows.append(f)\n    return rows\n\n\ndef meta_report(out_dir, top_k=200, csv_path=None, max_shards=None):\n    rows = fingerprint(out_dir, max_shards=max_shards)\n    if not rows:\n        print("No data. Run preprocessing first."); return None\n    keys = [k for k in rows[0] if k not in ("episode", "player")]\n    bal = np.array([r["final_balance"] for r in rows], dtype=np.float64)\n    order = np.argsort(-bal)\n    print(f"\\n===== KAGRICULTURE META REPORT ({len(rows):,} agent-episodes) =====")\n    print(f"final balance: max={bal.max():,.0f}  p95={np.percentile(bal,95):,.0f}  "\n          f"median={np.median(bal):,.0f}  mean={bal.mean():,.0f}")\n\n    # correlation of each numeric lever with final balance\n    print("\\n-- correlation of strategic levers with FINAL BALANCE --")\n    corrs = []\n    for k in keys:\n        if k in ("final_balance", "win"):\n            continue\n        x = np.array([r[k] for r in rows], dtype=np.float64)\n        if x.std() < 1e-9:\n            continue\n        c = np.corrcoef(x, bal)[0, 1]\n        corrs.append((k, c))\n    corrs.sort(key=lambda kv: -abs(kv[1]))\n    for k, c in corrs[:20]:\n        print(f"  {k:<20} r={c:+.3f}")\n\n    # top vs bottom cohort contrast\n    tk = order[:top_k]; bk = order[-top_k:]\n    print(f"\\n-- TOP {top_k} vs BOTTOM {top_k}: mean lever values --")\n    print(f"  {\'lever\':<20}{\'TOP\':>12}{\'BOTTOM\':>12}{\'ratio\':>10}")\n    for k in keys:\n        if k in ("final_balance", "win"):\n            continue\n        t = np.mean([rows[i][k] for i in tk])\n        b = np.mean([rows[i][k] for i in bk])\n        ratio = (t / b) if abs(b) > 1e-9 else float("inf")\n        print(f"  {k:<20}{t:>12.2f}{b:>12.2f}{ratio:>10.2f}")\n\n    if csv_path:\n        with open(csv_path, "w", newline="") as fh:\n            w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))\n            w.writeheader()\n            for r in rows:\n                w.writerow(r)\n        print(f"\\nPer-episode fingerprints written to {csv_path}")\n    return rows\n', 'kagri/preflight.py': '"""Preflight verification that runs before any long training job."""\nfrom __future__ import annotations\nimport os\nimport platform\nimport numpy as np\nimport torch\n\n\ndef inspect_processed(out_dir: str):\n    idx_path = os.path.join(out_dir, "index.npz")\n    if not os.path.exists(idx_path):\n        raise FileNotFoundError(idx_path)\n    with np.load(idx_path, allow_pickle=False) as idx:\n        shards = list(idx["shards"].astype(str))\n        indexed_turns = idx["counts"].astype(np.int64)\n    if len(shards) != len(indexed_turns):\n        raise RuntimeError("index.npz has mismatched shards/counts lengths")\n\n    total_turns = total_units = 0\n    unit_totals = []\n    for name, expected_turns in zip(shards, indexed_turns):\n        path = os.path.join(out_dir, name)\n        if not os.path.exists(path):\n            raise FileNotFoundError(path)\n        with np.load(path, allow_pickle=False) as z:\n            required = ["grids", "scalars", "unit_ops", "unit_counts",\n                        "market_qty", "market_bin", "rewards", "episode_id",\n                        "player", "step", "final_reward", "win"]\n            missing = [k for k in required if k not in z.files]\n            if missing:\n                raise RuntimeError(f"{name}: missing arrays {missing}")\n            turns = int(z["unit_counts"].shape[0])\n            units = int(z["unit_counts"].astype(np.int64).sum())\n            if turns != int(expected_turns):\n                raise RuntimeError(\n                    f"{name}: index counts={expected_turns}, actual turn rows={turns}"\n                )\n            if int(z["grids"].shape[0]) != turns or int(z["scalars"].shape[0]) != turns:\n                raise RuntimeError(f"{name}: state arrays are not aligned with unit_counts")\n            if int(z["unit_ops"].shape[0]) != units:\n                raise RuntimeError(f"{name}: unit_ops length {z[\'unit_ops\'].shape[0]} != {units}")\n            for k in ["market_qty", "market_bin", "rewards", "episode_id", "player", "step", "final_reward", "win"]:\n                if int(z[k].shape[0]) != turns:\n                    raise RuntimeError(f"{name}: {k} is not aligned with turn rows")\n            total_turns += turns\n            total_units += units\n            unit_totals.append(units)\n\n    return {\n        "shards": len(shards),\n        "turn_rows": total_turns,\n        "unit_examples": total_units,\n        "unit_totals": np.asarray(unit_totals, dtype=np.int64),\n    }\n\n\ndef print_environment():\n    print("=== KAGRI V3 PREFLIGHT ===")\n    print("Python:", platform.python_version())\n    print("PyTorch:", torch.__version__)\n    print("CUDA:", torch.version.cuda)\n    print("CUDA available:", torch.cuda.is_available())\n    if torch.cuda.is_available():\n        print("GPU count:", torch.cuda.device_count())\n        for i in range(torch.cuda.device_count()):\n            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")\n    print("===========================")\n', 'kagri/configs.py': 'import os\n"""Configuration for Kaggle free-tier training.\n\nThe defaults target the user\'s observed environment: 2 x Tesla T4, PyTorch 2.10,\nCUDA 12.8, with training left unattended.  The code performs a preflight check\nbefore consuming a long training run.\n"""\n\nDATASET_DIRS = [\n    f"/kaggle/input/datasets/kaggle/kaggriculture-episodes-2026-08-{d:02d}"\n    for d in range(10, 17)\n]\nDEFAULT_PROCESSED_DIR = os.environ.get(\n    "KAGRI_PROCESSED_DIR",\n    "/kaggle/input/notebooks/jominurk21cs1077/preprocessing/processed",\n)\nOUT_DIR = DEFAULT_PROCESSED_DIR\nWORK_DIR = "/kaggle/working/ckpt"\n\nPREPROCESS = dict(\n    dataset_dirs=DATASET_DIRS,\n    out_dir=OUT_DIR,\n    stride=1,\n    max_episodes=None,\n    workers=None,\n    compressed=True,       # safe default for Kaggle input storage\n)\n\n# Per-process batch for DDP. With 2 T4s this is a 4096 global batch.\nBC = dict(\n    out_dir=OUT_DIR,\n    work_dir=WORK_DIR,\n    dim=256,\n    dropout=0.10,\n    batch_size=2048,\n    workers=4,             # 4 workers/GPU; official PyTorch guidance is 2-4/GPU\n    prefetch=2,\n    persistent_workers=True,\n    use_ddp=True,\n    compile=False,         # deliberately off: observed Dynamo recompilation warnings\n    spatial_pool=2,        # preserve coarse 2x2 spatial layout instead of global pooling\n    lr=3e-4,\n    weight_decay=1e-4,\n    warmup=500,\n    epochs=6,\n    patience=3,\n    label_smoothing=0.05,\n    return_weighting=True,\n    market_w=0.5,\n    value_w=0.1,\n    clip=1.0,\n    val_frac=0.05,\n    val_max_examples=500_000,\n    seed=0,\n    log_every=200,\n    preflight=True,\n)\n\nIQL = dict(\n    out_dir=OUT_DIR, work_dir=WORK_DIR, dim=256, dropout=0.1,\n    batch_size=2048, workers=4, prefetch=2, use_ddp=True, compile=False,\n    lr=3e-4, weight_decay=1e-4, epochs=4,\n    expectile=0.7, awr_beta=3.0, gamma=0.99, clip=1.0,\n    val_frac=0.05, log_every=200,\n)\n\nDT = dict(\n    out_dir=OUT_DIR, work_dir=WORK_DIR, dim=256, n_layers=4, n_heads=8,\n    context=40, dropout=0.1, batch_size=64, workers=4,\n    prefetch=2, use_ddp=True, compile=False,\n    lr=1e-4, weight_decay=1e-4, warmup=500, epochs=6,\n    gamma=1.0, rtg_scale=1e5, label_smoothing=0.05, clip=1.0,\n    val_frac=0.05, log_every=100,\n)\n'}

for rel, code in SOURCES.items():
    p = WORK_ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(code, encoding="utf-8")

if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

print("Embedded Python files:", len(SOURCES))
for rel in sorted(SOURCES):
    print(" ", rel)


In [ ]:
# ============================================================
# 2. SOURCE COMPILE + IMPORT GATE
# ============================================================
for rel in sorted(SOURCES):
    if rel.endswith(".py"):
        py_compile.compile(str(WORK_ROOT / rel), doraise=True)

env = os.environ.copy()
env["PYTHONPATH"] = str(WORK_ROOT) + os.pathsep + env.get("PYTHONPATH", "")

# Explicitly import the actual classes/functions used by the trainer.
import torch
from kagri.models import BCPolicy
from kagri.encoding import expand_grid_torch
from kagri.datasets import UnitTransitionDataset, ShardBatchSampler, collate_unit

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("BCPolicy import: OK")
print("Dataset import: OK")
print("Encoding import: OK")
print("SOURCE / IMPORT GATE PASSED")


In [ ]:
# ============================================================
# 3. EXACT DATASET PATH + FULL PREFLIGHT
# ============================================================
if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Required processed dataset does not exist: {PROCESSED_DIR}"
    )

index_path = PROCESSED_DIR / "index.npz"
if not index_path.exists():
    raise FileNotFoundError(f"Missing required index: {index_path}")

from preflight import build_manifest, save_manifest

print("Scanning:", PROCESSED_DIR)
t0 = time.time()
manifest = build_manifest(
    str(PROCESSED_DIR),
    val_frac=0.05,
    seed=0,
)
save_manifest(manifest, str(MANIFEST_PATH))

print()
print("DATASET PREFLIGHT ===")
print("Shards:", manifest["shard_count"])
print("Turn rows:", f'{manifest["turn_rows"]:,}')
print("BC unit examples:", f'{manifest["unit_examples"]:,}')
print("Train unit examples:", f'{manifest["splits"]["train"]["unit_examples"]:,}')
print("Validation unit examples:", f'{manifest["splits"]["val"]["unit_examples"]:,}')
print("Train shards:", len(manifest["splits"]["train"]["shards"]))
print("Validation shards:", len(manifest["splits"]["val"]["shards"]))
print("Signature:", manifest["signature"])
print(f"Preflight elapsed: {(time.time()-t0)/60:.2f} min")
print("=========================")

# Hard correctness assertions based on the dataset you already measured.
assert manifest["shard_count"] == 4838, (
    f"Expected 4838 shards from the supplied dataset, got {manifest['shard_count']}"
)
assert manifest["turn_rows"] == 6966720, (
    f"Expected 6,966,720 turn rows, got {manifest['turn_rows']}"
)
assert manifest["unit_examples"] == 62868898, (
    f"Expected 62,868,898 BC unit examples, got {manifest['unit_examples']}"
)

print("DATASET CARDINALITY GATE PASSED")


In [ ]:
# ============================================================
# 4. REAL BATCH + MODEL FORWARD/BACKWARD SMOKE TEST
# ============================================================
from torch.utils.data import DataLoader

train_ds = UnitTransitionDataset(
    str(MANIFEST_PATH), "train", cache_capacity=1
)

sampler = ShardBatchSampler(
    train_ds,
    batch_size=CONFIG["batch_size_per_gpu"],
    shuffle=False,
    drop_last=True,
    seed=0,
    rank=0,
    world_size=1,
)

loader = DataLoader(
    train_ds,
    batch_sampler=sampler,
    num_workers=0,
    collate_fn=collate_unit,
    pin_memory=True,
)

batch = next(iter(loader))

print("Real batch:")
for k, v in batch.items():
    print(f"  {k:16s} {tuple(v.shape)} {v.dtype}")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = BCPolicy(dim=CONFIG["dim"], dropout=CONFIG["dropout"]).to(device)

param_count = sum(p.numel() for p in model.parameters())
print("Parameters:", param_count)

assert param_count == 477197, f"Unexpected parameter count: {param_count}"

# Match the actual trainer data flow.
grid_codes = batch["grid_codes"].to(device, non_blocking=True)
scalar = batch["scalar"].to(device, non_blocking=True)
unit_feat = batch["unit_feat"].to(device, non_blocking=True)
unit_op = batch["unit_op"].to(device, non_blocking=True)
market_qty = batch["market_qty"].to(device, non_blocking=True)
market_bin = batch["market_bin"].to(device, non_blocking=True)

model.train()
with torch.autocast(
    device_type="cuda" if device.type == "cuda" else "cpu",
    dtype=torch.float16 if device.type == "cuda" else torch.bfloat16,
):
    out = model(grid_codes, scalar, unit_feat)

# The exact output contract is checked by the trainer itself as well.
print("Forward output type:", type(out))
if isinstance(out, dict):
    print("Output keys:", sorted(out.keys()))
elif isinstance(out, (tuple, list)):
    print("Output count:", len(out))

# Backward through a real scalar from the returned output.
if isinstance(out, dict):
    candidates = [v for v in out.values() if torch.is_tensor(v) and v.numel() > 0]
    loss_smoke = sum(v.float().mean() for v in candidates)
elif isinstance(out, (tuple, list)):
    loss_smoke = sum(v.float().mean() for v in out if torch.is_tensor(v) and v.numel() > 0)
else:
    loss_smoke = out.float().mean()

loss_smoke.backward()

assert any(p.grad is not None for p in model.parameters())
print("FORWARD/BACKWARD SMOKE TEST PASSED")


In [ ]:
# ============================================================
# 5. SAVE CONFIGURATION USED FOR RESUME VALIDATION
# ============================================================
CONFIG_PATH.write_text(
    json.dumps(CONFIG, indent=2, sort_keys=True),
    encoding="utf-8",
)

print(CONFIG_PATH.read_text())


## 6. FINAL 2×T4 DDP GATE

This launches a **short DDP smoke test only**. It is not the training run.

It must succeed before the final training cell is allowed to start.


In [ ]:
# ============================================================
# 6. 2×T4 DDP SMOKE TEST
# ============================================================
if torch.cuda.device_count() != 2:
    raise RuntimeError(
        f"Expected exactly 2 GPUs for this unattended configuration; "
        f"detected {torch.cuda.device_count()}."
    )

env = os.environ.copy()
env["PYTHONPATH"] = str(WORK_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
env["KAGRI_CONFIG"] = str(CONFIG_PATH)
env["KAGRI_MANIFEST"] = str(MANIFEST_PATH)
env["KAGRI_WORK"] = str(CKPT_DIR)

p = subprocess.run(
    [
        "torchrun",
        "--standalone",
        "--nproc_per_node=2",
        str(WORK_ROOT / "ddp_smoke.py"),
    ],
    cwd=str(WORK_ROOT),
    env=env,
    text=True,
)

if p.returncode != 0:
    raise RuntimeError(
        "DDP smoke test failed. Training has NOT been started."
    )

print("2×T4 DDP SMOKE TEST PASSED")


# 7. FINAL UNATTENDED TRAINING

**This is the only long-running cell.**

At this point:

- the fixed dataset path has been verified;
- all 4,838 shards have passed preflight;
- 62,868,898 BC unit examples have been confirmed;
- all embedded source files compile/import;
- the real data batch has been tested;
- the model has completed forward/backward;
- 2×GPU DDP has passed its smoke test.

The trainer owns checkpointing and resume. If Kaggle later interrupts the session, rerunning the notebook will reconstruct the same manifest/config and resume only when the checkpoint signature matches.


In [ ]:
# ============================================================
# 7. START FINAL UNATTENDED TRAINING
# ============================================================
print("FINAL TRAINING GATE")
print("Processed:", PROCESSED_DIR)
print("Shards:", manifest["shard_count"])
print("Turn rows:", f'{manifest["turn_rows"]:,}')
print("BC examples:", f'{manifest["unit_examples"]:,}')
print("Train:", f'{manifest["splits"]["train"]["unit_examples"]:,}')
print("Val:", f'{manifest["splits"]["val"]["unit_examples"]:,}')
print("GPUs:", torch.cuda.device_count())
print("Effective batch:", CONFIG["batch_size_per_gpu"] * torch.cuda.device_count())
print("Checkpoint:", CKPT_DIR)
print()
print("Launching final unattended training...")

p = subprocess.run(
    [
        "torchrun",
        "--standalone",
        "--nproc_per_node=2",
        str(WORK_ROOT / "train_bc.py"),
    ],
    cwd=str(WORK_ROOT),
    env=env,
    text=True,
)

if p.returncode != 0:
    raise RuntimeError(
        f"Training exited with status {p.returncode}. "
        "Inspect the training logs above; no automatic configuration change is made."
    )

print("TRAINING PROCESS COMPLETED SUCCESSFULLY")
